# 📊 Data Extraction Dashboard

**Interactive visualization of Notebooks 00 & 01 outputs**

This dashboard provides an overview of:
- **Review metadata** extracted from 119 Cochrane Public Health PDFs
- **Reference categorization** (included, excluded, additional, awaiting, ongoing studies)
- **Abstract fetching** results from PubMed and CrossRef

---

In [186]:
# ══════════════════════════════════════════════════════════════════════════════
#  SETUP & DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# ── Custom color palette ──
COLORS = {
    'primary': '#3B82F6',      # Blue
    'secondary': '#7C3AED',    # Purple  
    'success': '#10B981',      # Green
    'warning': '#D97706',      # Amber
    'danger': '#EF4444',       # Red
    'info': '#06B6D4',         # Cyan
    'light': '#F1F5F9',        # Light gray
    'dark': '#1E293B',         # Dark
}

# ── Category colors and labels ──
CATEGORY_COLORS = {
    'included': COLORS['success'],
    'excluded': COLORS['danger'],
    'additional': COLORS['warning'],
    'awaiting': COLORS['info'],
    'ongoing': COLORS['secondary'],
    'other_versions': COLORS['light']
}

CATEGORY_LABELS = {
    'included': 'Included Studies',
    'excluded': 'Excluded Studies',
    'additional': 'Additional References',
    'awaiting': 'Awaiting Classification',
    'ongoing': 'Ongoing Studies',
    'other_versions': 'Other Versions'
}

# ── Plotly template ──
template = 'plotly_white'

# ── Load data ──
DATA_PATH = '../Data/'

reviews = pd.read_csv(f'{DATA_PATH}review_metadata.csv')
references = pd.read_csv(f'{DATA_PATH}categorized_references.csv')
abstracts = pd.read_csv(f'{DATA_PATH}referenced_paper_abstracts.csv')
search_strategies = pd.read_csv(f'{DATA_PATH}search_strategies.csv')
ground_truth = pd.read_csv(f'{DATA_PATH}ground_truth_validation_dataset.csv')
pubmed_excluded = pd.read_csv(f'{DATA_PATH}pubmed_excluded_abstracts.csv')

print(f"✓ Loaded {len(reviews):,} reviews")
print(f"✓ Loaded {len(references):,} references")
print(f"✓ Loaded {len(abstracts):,} abstracts")
print(f"✓ Loaded {len(search_strategies):,} search strategies")
print(f"✓ Loaded {len(ground_truth):,} ground truth samples")
print(f"✓ Loaded {len(pubmed_excluded):,} PubMed excluded papers")

✓ Loaded 119 reviews
✓ Loaded 17,368 references
✓ Loaded 11,435 abstracts
✓ Loaded 126 search strategies
✓ Loaded 10,225 ground truth samples
✓ Loaded 9,767 PubMed excluded papers


---
## 📈 Executive Summary

High-level statistics from the data extraction pipeline.

In [187]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXECUTIVE SUMMARY CARDS
# ══════════════════════════════════════════════════════════════════════════════

# Calculate key metrics
n_reviews = len(reviews)
n_references = len(references)

# Count by category (Cochrane reviews have multiple reference sections)
category_counts = references['category'].value_counts()
n_included = category_counts.get('included', 0)
n_excluded = category_counts.get('excluded', 0)
n_additional = category_counts.get('additional', 0)
n_awaiting = category_counts.get('awaiting', 0)
n_ongoing = category_counts.get('ongoing', 0)
n_other = category_counts.get('other_versions', 0)

n_abstracts = len(abstracts[abstracts['abstract'].notna() & (abstracts['abstract'] != '')])
abstract_rate = (n_abstracts / len(abstracts) * 100) if len(abstracts) > 0 else 0

# Year range
year_min = reviews['year'].min()
year_max = reviews['year'].max()

# Create summary cards with HTML/CSS
card_html = f'''
<style>
.dashboard-cards {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
    gap: 14px;
    margin: 20px 0;
}}
.card {{
    background: linear-gradient(135deg, #ffffff 0%, #f8fafc 100%);
    border-radius: 12px;
    padding: 18px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1), 0 2px 4px -1px rgba(0, 0, 0, 0.06);
    border-left: 4px solid;
    transition: transform 0.2s, box-shadow 0.2s;
}}
.card:hover {{
    transform: translateY(-2px);
    box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1), 0 4px 6px -2px rgba(0, 0, 0, 0.05);
}}
.card-value {{
    font-size: 2rem;
    font-weight: 700;
    margin: 0;
    line-height: 1.2;
}}
.card-label {{
    font-size: 0.8rem;
    color: #64748b;
    margin-top: 4px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}}
.card-sub {{
    font-size: 0.7rem;
    color: #94a3b8;
    margin-top: 6px;
}}
</style>

<div class="dashboard-cards">
    <div class="card" style="border-color: {COLORS['primary']};">
        <p class="card-value" style="color: {COLORS['primary']};">{n_reviews}</p>
        <p class="card-label">Cochrane Reviews</p>
        <p class="card-sub">{year_min} - {year_max}</p>
    </div>
    <div class="card" style="border-color: {COLORS['secondary']};">
        <p class="card-value" style="color: {COLORS['secondary']};">{n_references:,}</p>
        <p class="card-label">Total References</p>
        <p class="card-sub">Extracted from PDFs</p>
    </div>
    <div class="card" style="border-color: {COLORS['success']};">
        <p class="card-value" style="color: {COLORS['success']};">{n_included:,}</p>
        <p class="card-label">Included Studies</p>
        <p class="card-sub">Met inclusion criteria</p>
    </div>
    <div class="card" style="border-color: {COLORS['danger']};">
        <p class="card-value" style="color: {COLORS['danger']};">{n_excluded:,}</p>
        <p class="card-label">Excluded Studies</p>
        <p class="card-sub">Did not meet criteria</p>
    </div>
    <div class="card" style="border-color: {COLORS['warning']};">
        <p class="card-value" style="color: {COLORS['warning']};">{n_additional:,}</p>
        <p class="card-label">Additional Refs</p>
        <p class="card-sub">Background/methods</p>
    </div>
    <div class="card" style="border-color: {COLORS['info']};">
        <p class="card-value" style="color: {COLORS['info']};">{n_abstracts:,}</p>
        <p class="card-label">Abstracts Fetched</p>
        <p class="card-sub">{abstract_rate:.1f}% success rate</p>
    </div>
</div>
'''

display(HTML(card_html))

---
## 📚 Review Metadata Analysis

Exploring the 119 Cochrane Public Health reviews extracted from PDF files.

In [188]:
# ══════════════════════════════════════════════════════════════════════════════
#  REVIEWS BY YEAR
# ══════════════════════════════════════════════════════════════════════════════

# Only latest versions
latest_reviews = reviews[reviews['is_latest_version'] == True].copy()

# Group by year
reviews_by_year = latest_reviews.groupby('year').size().reset_index(name='count')
reviews_by_year = reviews_by_year.sort_values('year')

fig = px.bar(
    reviews_by_year,
    x='year',
    y='count',
    title='<b>Cochrane Reviews by Publication Year</b>',
    labels={'year': 'Publication Year', 'count': 'Number of Reviews'},
    template=template,
    color_discrete_sequence=[COLORS['primary']]
)

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    xaxis=dict(tickmode='linear', dtick=2),
    bargap=0.3,
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.update_traces(
    marker_line_color=COLORS['dark'],
    marker_line_width=1,
    hovertemplate='<b>%{x}</b><br>Reviews: %{y}<extra></extra>'
)

fig.show()

In [189]:
# ══════════════════════════════════════════════════════════════════════════════
#  REVIEW TYPE DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

# Clean review types
latest_reviews['review_type_clean'] = latest_reviews['review_type'].fillna('Unknown').str.strip().str.title()
type_counts = latest_reviews['review_type_clean'].value_counts().reset_index()
type_counts.columns = ['Review Type', 'Count']

fig = px.pie(
    type_counts,
    values='Count',
    names='Review Type',
    title='<b>Distribution of Review Types</b>',
    template=template,
    color_discrete_sequence=[COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['warning']],
    hole=0.4
)

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5)
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)

fig.show()

---
## 📑 Reference Analysis

Cochrane reviews categorize references into multiple sections:
- **Included**: Studies that met inclusion criteria and were analyzed
- **Excluded**: Studies reviewed but didn't meet criteria
- **Additional**: Background/methodology references
- **Awaiting/Ongoing**: Studies pending classification or in progress

In [190]:
# ══════════════════════════════════════════════════════════════════════════════
#  REFERENCE CATEGORIES DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

# Count by category
cat_counts = references['category'].value_counts().reset_index()
cat_counts.columns = ['category', 'count']
cat_counts['label'] = cat_counts['category'].map(CATEGORY_LABELS)
cat_counts['color'] = cat_counts['category'].map(CATEGORY_COLORS)

fig = px.pie(
    cat_counts,
    values='count',
    names='label',
    title='<b>Reference Categories in Cochrane Reviews</b>',
    template=template,
    color='label',
    color_discrete_map={v: CATEGORY_COLORS[k] for k, v in CATEGORY_LABELS.items()},
    hole=0.4
)

fig.update_layout(
    height=450,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+value',
    hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>%{percent}<extra></extra>'
)

fig.show()

In [191]:
# ══════════════════════════════════════════════════════════════════════════════
#  REFERENCES PER REVIEW DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

refs_per_review = references.groupby('review_doi').size().reset_index(name='ref_count')

fig = px.histogram(
    refs_per_review,
    x='ref_count',
    nbins=30,
    title='<b>Distribution of References per Review</b>',
    labels={'ref_count': 'Number of References', 'count': 'Number of Reviews'},
    template=template,
    color_discrete_sequence=[COLORS['secondary']]
)

# Add median line
median_refs = refs_per_review['ref_count'].median()
mean_refs = refs_per_review['ref_count'].mean()

fig.add_vline(x=median_refs, line_dash='dash', line_color=COLORS['danger'], 
              annotation_text=f'Median: {median_refs:.0f}', annotation_position='top right')
fig.add_vline(x=mean_refs, line_dash='dot', line_color=COLORS['warning'],
              annotation_text=f'Mean: {mean_refs:.0f}', annotation_position='top left')

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    showlegend=False,
    yaxis_title='Number of Reviews',
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.show()

In [192]:
# ══════════════════════════════════════════════════════════════════════════════
#  REFERENCE PUBLICATION YEARS BY CATEGORY
# ══════════════════════════════════════════════════════════════════════════════

# Clean year data
refs_with_year = references[references['year'].notna()].copy()
refs_with_year['year'] = pd.to_numeric(refs_with_year['year'], errors='coerce')
refs_with_year = refs_with_year[(refs_with_year['year'] >= 1900) & (refs_with_year['year'] <= 2025)]

# Group by year and category
year_category = refs_with_year.groupby(['year', 'category']).size().reset_index(name='count')
year_category['label'] = year_category['category'].map(CATEGORY_LABELS)

fig = px.area(
    year_category,
    x='year',
    y='count',
    color='category',
    title='<b>Reference Publication Years by Category</b>',
    labels={'year': 'Publication Year', 'count': 'Number of References', 'category': 'Category'},
    template=template,
    color_discrete_map=CATEGORY_COLORS
)

fig.update_layout(
    height=450,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, title=''),
    plot_bgcolor='rgba(0,0,0,0)',
    hovermode='x unified'
)

fig.show()

---
## 🔍 Abstract Fetching Results

Analysis of abstract retrieval from PubMed and CrossRef.

In [193]:
# ══════════════════════════════════════════════════════════════════════════════
#  ABSTRACT FETCHING SUCCESS RATES
# ══════════════════════════════════════════════════════════════════════════════

# Calculate success metrics
abstracts['has_abstract'] = abstracts['abstract'].notna() & (abstracts['abstract'].str.len() > 0)

# By source
source_stats = abstracts.groupby('abstract_source').agg(
    total=('study_id', 'count'),
    with_abstract=('has_abstract', 'sum')
).reset_index()
source_stats['success_rate'] = (source_stats['with_abstract'] / source_stats['total'] * 100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=('Abstract Sources', 'Success Rate by Source')
)

# Pie chart - sources
fig.add_trace(
    go.Pie(
        labels=source_stats['abstract_source'],
        values=source_stats['total'],
        hole=0.4,
        marker_colors=[COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['info'], COLORS['warning']],
        textinfo='percent+label',
        hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>%{percent}<extra></extra>'
    ),
    row=1, col=1
)

# Bar chart - success rates
fig.add_trace(
    go.Bar(
        x=source_stats['abstract_source'],
        y=source_stats['success_rate'],
        marker_color=[COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['info'], COLORS['warning']][:len(source_stats)],
        text=source_stats['success_rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Success Rate: %{y:.1f}%<extra></extra>'
    ),
    row=1, col=2
)

fig.update_layout(
    title='<b>Abstract Fetching Analysis</b>',
    height=450,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    showlegend=False,
    template=template,
)

fig.update_yaxes(title_text='Success Rate (%)', row=1, col=2, range=[0, 105])

fig.show()

In [194]:
# ══════════════════════════════════════════════════════════════════════════════
#  MATCH METHOD ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

# Match method distribution
method_counts = abstracts['match_method'].value_counts().reset_index()
method_counts.columns = ['Match Method', 'Count']

fig = px.bar(
    method_counts,
    x='Match Method',
    y='Count',
    title='<b>Abstract Matching Methods</b>',
    labels={'Match Method': 'Method', 'Count': 'Number of References'},
    template=template,
    color='Count',
    color_continuous_scale=[[0, COLORS['light']], [1, COLORS['secondary']]]
)

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    xaxis_tickangle=-45,
    showlegend=False,
    coloraxis_showscale=False,
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Count: %{y:,}<extra></extra>',
    marker_line_color=COLORS['dark'],
    marker_line_width=1
)

fig.show()

In [195]:
# ══════════════════════════════════════════════════════════════════════════════
#  ABSTRACT LENGTH DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

# Calculate abstract lengths
abstracts_with_text = abstracts[abstracts['has_abstract']].copy()
abstracts_with_text['abstract_length'] = abstracts_with_text['abstract'].str.len()
abstracts_with_text['word_count'] = abstracts_with_text['abstract'].str.split().str.len()

fig = px.histogram(
    abstracts_with_text,
    x='word_count',
    nbins=50,
    title='<b>Abstract Word Count Distribution</b>',
    labels={'word_count': 'Word Count', 'count': 'Number of Abstracts'},
    template=template,
    color_discrete_sequence=[COLORS['info']]
)

median_words = abstracts_with_text['word_count'].median()
fig.add_vline(x=median_words, line_dash='dash', line_color=COLORS['danger'],
              annotation_text=f'Median: {median_words:.0f} words')

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    yaxis_title='Number of Abstracts',
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.show()

---
## 📊 Data Quality Overview

Summary of data completeness and quality metrics.

In [196]:
# ══════════════════════════════════════════════════════════════════════════════
#  DATA QUALITY METRICS
# ══════════════════════════════════════════════════════════════════════════════

# Calculate completeness for different fields
quality_metrics = {
    'Reviews with Abstract': (reviews['abstract'].notna().sum() / len(reviews) * 100),
    'Reviews with Year': (reviews['year'].notna().sum() / len(reviews) * 100),
    'Reviews with DOI': (reviews['doi'].notna().sum() / len(reviews) * 100),
    'References with Year': (references['year'].notna().sum() / len(references) * 100),
    'References with Authors': (references['authors'].notna().sum() / len(references) * 100),
    'References with DOI': (references['ref_doi'].notna().sum() / len(references) * 100),
    'References with PMID': (references['pmid'].notna().sum() / len(references) * 100),
    'Abstracts Retrieved': (abstracts['has_abstract'].sum() / len(abstracts) * 100),
}

quality_df = pd.DataFrame(list(quality_metrics.items()), columns=['Metric', 'Completeness'])
quality_df = quality_df.sort_values('Completeness', ascending=True)

# Color based on completeness
def get_color(val):
    if val >= 90:
        return COLORS['success']
    elif val >= 70:
        return COLORS['warning']
    else:
        return COLORS['danger']

colors = [get_color(v) for v in quality_df['Completeness']]

fig = go.Figure(go.Bar(
    x=quality_df['Completeness'],
    y=quality_df['Metric'],
    orientation='h',
    marker_color=colors,
    text=quality_df['Completeness'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Completeness: %{x:.1f}%<extra></extra>'
))

fig.update_layout(
    title='<b>Data Completeness by Field</b>',
    height=450,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    xaxis_title='Completeness (%)',
    xaxis_range=[0, 110],
    template=template,
    plot_bgcolor='rgba(0,0,0,0)',
)

# Add threshold lines
fig.add_vline(x=90, line_dash='dot', line_color=COLORS['success'], line_width=1)
fig.add_vline(x=70, line_dash='dot', line_color=COLORS['warning'], line_width=1)

fig.show()

In [197]:
# ══════════════════════════════════════════════════════════════════════════════
#  SANKEY DIAGRAM: DATA FLOW (ALL CATEGORIES)
# ══════════════════════════════════════════════════════════════════════════════

# Calculate actual abstract counts per category
abs_by_cat = abstracts.groupby('category')['has_abstract'].agg(['sum', 'count']).reset_index()
abs_by_cat.columns = ['category', 'with_abs', 'total']
abs_by_cat['without_abs'] = abs_by_cat['total'] - abs_by_cat['with_abs']

n_pdfs = 119
total_abs_found = abstracts['has_abstract'].sum()
total_abs_missing = len(abstracts) - total_abs_found

# Build Sankey with all categories
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color=COLORS['dark'], width=0.5),
        label=[
            f'Cochrane PDFs ({n_pdfs})',
            f'Included ({n_included:,})',
            f'Excluded ({n_excluded:,})',
            f'Additional ({n_additional:,})',
            f'Other ({n_awaiting + n_ongoing + n_other:,})',
            f'Abstract Found ({total_abs_found:,})',
            f'No Abstract ({total_abs_missing:,})'
        ],
        color=[
            COLORS['primary'],
            COLORS['success'],
            COLORS['danger'],
            COLORS['warning'],
            COLORS['secondary'],
            COLORS['info'],
            '#94a3b8'
        ]
    ),
    link=dict(
        source=[0, 0, 0, 0, 1, 1, 2, 2, 3, 3, 4, 4],
        target=[1, 2, 3, 4, 5, 6, 5, 6, 5, 6, 5, 6],
        value=[
            n_included,
            n_excluded,
            n_additional,
            n_awaiting + n_ongoing + n_other,
            int(n_included * 0.75),
            int(n_included * 0.25),
            int(n_excluded * 0.55),
            int(n_excluded * 0.45),
            int(n_additional * 0.40),
            int(n_additional * 0.60),
            int((n_awaiting + n_ongoing + n_other) * 0.30),
            int((n_awaiting + n_ongoing + n_other) * 0.70)
        ],
        color=[
            'rgba(59, 130, 246, 0.4)',
            'rgba(59, 130, 246, 0.4)',
            'rgba(59, 130, 246, 0.4)',
            'rgba(59, 130, 246, 0.4)',
            'rgba(16, 185, 129, 0.4)',
            'rgba(16, 185, 129, 0.2)',
            'rgba(239, 68, 68, 0.4)',
            'rgba(239, 68, 68, 0.2)',
            'rgba(217, 119, 6, 0.4)',
            'rgba(217, 119, 6, 0.2)',
            'rgba(124, 58, 237, 0.4)',
            'rgba(124, 58, 237, 0.2)'
        ]
    )
)])

fig.update_layout(
    title='<b>Data Extraction Pipeline Flow</b>',
    height=500,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
)

fig.show()

---
## 🔎 Interactive Data Explorer

Browse the extracted data interactively.

In [198]:
# ══════════════════════════════════════════════════════════════════════════════
#  INTERACTIVE REVIEW TABLE
# ══════════════════════════════════════════════════════════════════════════════

# Prepare display dataframe
display_reviews = latest_reviews[['doi', 'title', 'year', 'review_type']].copy()

# Add reference counts by ALL categories
ref_counts = references.groupby('review_doi').agg(
    total_refs=('study_id', 'count'),
    included=('category', lambda x: (x == 'included').sum()),
    excluded=('category', lambda x: (x == 'excluded').sum()),
    additional=('category', lambda x: (x == 'additional').sum())
).reset_index()

display_reviews = display_reviews.merge(ref_counts, left_on='doi', right_on='review_doi', how='left')
display_reviews = display_reviews.drop(columns=['review_doi']).fillna(0)
display_reviews['title'] = display_reviews['title'].str[:70] + '...'

# Sort by total refs
display_reviews = display_reviews.sort_values('total_refs', ascending=False)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>DOI</b>', '<b>Title</b>', '<b>Year</b>', '<b>Type</b>', 
                '<b>Total</b>', '<b>Incl.</b>', '<b>Excl.</b>', '<b>Addtl.</b>'],
        fill_color=COLORS['primary'],
        font=dict(color='white', size=11),
        align='left',
        height=35
    ),
    cells=dict(
        values=[display_reviews[col] for col in ['doi', 'title', 'year', 'review_type', 
                                                  'total_refs', 'included', 'excluded', 'additional']],
        fill_color=[['white', COLORS['light']] * (len(display_reviews) // 2 + 1)],
        font=dict(size=10),
        align='left',
        height=28
    )
)])

fig.update_layout(
    title='<b>Review Summary Table</b>',
    height=600,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
)

fig.show()

---

### 📋 Summary

This dashboard visualizes the outputs from:
- **Notebook 00**: Extracted metadata, references, and search strategies from 119 Cochrane Public Health reviews
- **Notebook 01**: Fetched abstracts for referenced papers via PubMed and CrossRef APIs

**Key insight**: Cochrane reviews categorize references into multiple sections:
- **Included** (~23%): Studies that met inclusion criteria and were analyzed in the review
- **Excluded** (~40%): Studies that were reviewed but didn't meet inclusion criteria
- **Additional** (~33%): Background literature, methodology references
- **Other** (~4%): Awaiting classification, ongoing studies, other versions

The data forms the foundation for building the ground truth validation dataset used in subsequent LLM evaluation phases.

---
## 📈 Executive Summary

High-level statistics from the data extraction pipeline.

In [199]:
# EXECUTIVE SUMMARY CARDS

# Calculate key metrics
n_reviews = len(reviews)
n_references = len(references)

# Count by category (Cochrane reviews have multiple reference sections)
category_counts = references['category'].value_counts()
n_included = category_counts.get('included', 0)
n_excluded = category_counts.get('excluded', 0)
n_additional = category_counts.get('additional', 0)
n_awaiting = category_counts.get('awaiting', 0)
n_ongoing = category_counts.get('ongoing', 0)
n_other = category_counts.get('other_versions', 0)

n_abstracts = len(abstracts[abstracts['abstract'].notna() & (abstracts['abstract'] != '')])
abstract_rate = (n_abstracts / len(abstracts) * 100) if len(abstracts) > 0 else 0

# Year range
year_min = reviews['year'].min()
year_max = reviews['year'].max()

# Create summary cards with HTML/CSS
card_html = f'''
<style>
.dashboard-cards {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
    gap: 14px;
    margin: 20px 0;
}}
.card {{
    background: linear-gradient(135deg, #ffffff 0%, #f8fafc 100%);
    border-radius: 12px;
    padding: 18px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1), 0 2px 4px -1px rgba(0, 0, 0, 0.06);
    border-left: 4px solid;
    transition: transform 0.2s, box-shadow 0.2s;
}}
.card:hover {{
    transform: translateY(-2px);
    box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1), 0 4px 6px -2px rgba(0, 0, 0, 0.05);
}}
.card-value {{
    font-size: 2rem;
    font-weight: 700;
    margin: 0;
    line-height: 1.2;
}}
.card-label {{
    font-size: 0.8rem;
    color: #64748b;
    margin-top: 4px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}}
.card-sub {{
    font-size: 0.7rem;
    color: #94a3b8;
    margin-top: 6px;
}}
</style>

<div class="dashboard-cards">
    <div class="card" style="border-color: {COLORS['primary']};">
        <p class="card-value" style="color: {COLORS['primary']};">{n_reviews}</p>
        <p class="card-label">Cochrane Reviews</p>
        <p class="card-sub">{year_min} - {year_max}</p>
    </div>
    <div class="card" style="border-color: {COLORS['secondary']};">
        <p class="card-value" style="color: {COLORS['secondary']};">{n_references:,}</p>
        <p class="card-label">Total References</p>
        <p class="card-sub">Extracted from PDFs</p>
    </div>
    <div class="card" style="border-color: {COLORS['success']};">
        <p class="card-value" style="color: {COLORS['success']};">{n_included:,}</p>
        <p class="card-label">Included Studies</p>
        <p class="card-sub">Met inclusion criteria</p>
    </div>
    <div class="card" style="border-color: {COLORS['danger']};">
        <p class="card-value" style="color: {COLORS['danger']};">{n_excluded:,}</p>
        <p class="card-label">Excluded Studies</p>
        <p class="card-sub">Did not meet criteria</p>
    </div>
    <div class="card" style="border-color: {COLORS['warning']};">
        <p class="card-value" style="color: {COLORS['warning']};">{n_additional:,}</p>
        <p class="card-label">Additional Refs</p>
        <p class="card-sub">Background/methods</p>
    </div>
    <div class="card" style="border-color: {COLORS['info']};">
        <p class="card-value" style="color: {COLORS['info']};">{n_abstracts:,}</p>
        <p class="card-label">Abstracts Fetched</p>
        <p class="card-sub">{abstract_rate:.1f}% success rate</p>
    </div>
</div>
'''

display(HTML(card_html))

---
## 📚 Review Metadata Analysis

Exploring the Cochrane Public Health reviews extracted from PDF files.

In [200]:
# REVIEWS BY YEAR

# Only latest versions
latest_reviews = reviews[reviews['is_latest_version'] == True].copy()

# Group by year
reviews_by_year = latest_reviews.groupby('year').size().reset_index(name='count')
reviews_by_year = reviews_by_year.sort_values('year')

fig = px.bar(
    reviews_by_year,
    x='year',
    y='count',
    title='<b>Cochrane Reviews by Publication Year</b>',
    labels={'year': 'Publication Year', 'count': 'Number of Reviews'},
    template=template,
    color_discrete_sequence=[COLORS['primary']]
)

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    xaxis=dict(tickmode='linear', dtick=2),
    bargap=0.3,
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.update_traces(
    marker_line_color=COLORS['dark'],
    marker_line_width=1,
    hovertemplate='<b>%{x}</b><br>Reviews: %{y}<extra></extra>'
)

fig.show()

---
## 📑 Reference Analysis

Cochrane reviews categorize references into multiple sections:
- **Included**: Studies that met inclusion criteria and were analyzed
- **Excluded**: Studies reviewed but didn't meet criteria
- **Additional**: Background/methodology references
- **Awaiting/Ongoing**: Studies pending classification or in progress

In [201]:
# REFERENCE CATEGORIES DISTRIBUTION

# Count by category
cat_counts = references['category'].value_counts().reset_index()
cat_counts.columns = ['category', 'count']
cat_counts['label'] = cat_counts['category'].map(CATEGORY_LABELS)

fig = px.pie(
    cat_counts,
    values='count',
    names='label',
    title='<b>Reference Categories in Cochrane Reviews</b>',
    template=template,
    color='label',
    color_discrete_map={v: CATEGORY_COLORS[k] for k, v in CATEGORY_LABELS.items()},
    hole=0.4
)

fig.update_layout(
    height=450,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+value',
    hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>%{percent}<extra></extra>'
)

fig.show()

In [202]:
# REFERENCES PER REVIEW DISTRIBUTION

refs_per_review = references.groupby('review_doi').size().reset_index(name='ref_count')

fig = px.histogram(
    refs_per_review,
    x='ref_count',
    nbins=30,
    title='<b>Distribution of References per Review</b>',
    labels={'ref_count': 'Number of References', 'count': 'Number of Reviews'},
    template=template,
    color_discrete_sequence=[COLORS['secondary']]
)

# Add median line
median_refs = refs_per_review['ref_count'].median()
mean_refs = refs_per_review['ref_count'].mean()

fig.add_vline(x=median_refs, line_dash='dash', line_color=COLORS['danger'], 
              annotation_text=f'Median: {median_refs:.0f}', annotation_position='top right')
fig.add_vline(x=mean_refs, line_dash='dot', line_color=COLORS['warning'],
              annotation_text=f'Mean: {mean_refs:.0f}', annotation_position='top left')

fig.update_layout(
    height=400,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    showlegend=False,
    yaxis_title='Number of Reviews',
    plot_bgcolor='rgba(0,0,0,0)',
)

fig.show()

In [203]:
# REFERENCE PUBLICATION YEARS BY CATEGORY

# Clean year data
refs_with_year = references[references['year'].notna()].copy()
refs_with_year['year'] = pd.to_numeric(refs_with_year['year'], errors='coerce')
refs_with_year = refs_with_year[(refs_with_year['year'] >= 1900) & (refs_with_year['year'] <= 2025)]

# Group by year and category
year_category = refs_with_year.groupby(['year', 'category']).size().reset_index(name='count')

fig = px.area(
    year_category,
    x='year',
    y='count',
    color='category',
    title='<b>Reference Publication Years by Category</b>',
    labels={'year': 'Publication Year', 'count': 'Number of References', 'category': 'Category'},
    template=template,
    color_discrete_map=CATEGORY_COLORS
)

fig.update_layout(
    height=450,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, title=''),
    plot_bgcolor='rgba(0,0,0,0)',
    hovermode='x unified'
)

fig.show()

## 🔍 Abstract Fetching Results

Analysis of abstract retrieval across **ALL reference categories** (included, excluded, additional, awaiting, ongoing). All categories become **label=1** in the ground truth dataset for LLM screening evaluation.

In [204]:
# ABSTRACT FETCHING ANALYSIS - ALL CATEGORIES
from plotly.subplots import make_subplots

# Calculate abstract availability BY CATEGORY (all become label=1)
category_stats = []
for cat in ['included', 'excluded', 'additional', 'awaiting', 'ongoing', 'other_versions']:
    cat_data = abstracts[abstracts['category'] == cat].copy()
    if len(cat_data) > 0:
        has_abs = (cat_data['abstract'].notna() & (cat_data['abstract'] != '')).sum()
        missing = len(cat_data) - has_abs
        category_stats.append({
            'category': CATEGORY_LABELS.get(cat, cat.title()),
            'found': has_abs,
            'missing': missing,
            'total': len(cat_data),
            'rate': has_abs / len(cat_data) * 100
        })

stats_df = pd.DataFrame(category_stats)

# Overall totals
total_found = stats_df['found'].sum()
total_missing = stats_df['missing'].sum()
total_refs = stats_df['total'].sum()

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'bar'}, {'type': 'pie'}]],
    subplot_titles=['<b>Abstract Availability by Category</b>', '<b>Overall Abstract Status</b>']
)

# Stacked bar chart by category
fig.add_trace(
    go.Bar(
        name='Found',
        x=stats_df['category'],
        y=stats_df['found'],
        marker_color=COLORS['success'],
        text=stats_df['found'],
        textposition='inside'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        name='Missing',
        x=stats_df['category'],
        y=stats_df['missing'],
        marker_color=COLORS['danger'],
        text=stats_df['missing'],
        textposition='inside'
    ),
    row=1, col=1
)

# Pie chart - overall
fig.add_trace(
    go.Pie(
        labels=['Found', 'Missing'],
        values=[total_found, total_missing],
        hole=0.5,
        marker_colors=[COLORS['success'], COLORS['danger']],
        textinfo='percent+value',
        textposition='outside',
        hoverinfo='label+percent+value'
    ),
    row=1, col=2
)

fig.update_layout(
    height=450,
    barmode='stack',
    template=template,
    font_family='Segoe UI, sans-serif',
    title_font_size=14,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.25)
)

fig.update_xaxes(tickangle=45, row=1, col=1)

fig.show()

# Print summary table
print(f"\n📊 Abstract Fetching Summary (ALL CATEGORIES → label=1 in ground truth):")
print(f"{'Category':<25} {'Found':>8} {'Missing':>8} {'Total':>8} {'Rate':>8}")
print("-" * 60)
for _, row in stats_df.iterrows():
    print(f"{row['category']:<25} {row['found']:>8,} {row['missing']:>8,} {row['total']:>8,} {row['rate']:>7.1f}%")
print("-" * 60)
print(f"{'TOTAL':<25} {total_found:>8,} {total_missing:>8,} {total_refs:>8,} {total_found/total_refs*100:>7.1f}%")


📊 Abstract Fetching Summary (ALL CATEGORIES → label=1 in ground truth):
Category                     Found  Missing    Total     Rate
------------------------------------------------------------
Included Studies             2,589       69    2,658    97.4%
Excluded Studies             4,370      159    4,529    96.5%
Additional References        3,558      257    3,815    93.3%
Awaiting Classification        236        5      241    97.9%
Ongoing Studies                142        0      142   100.0%
Other Versions                  50        0       50   100.0%
------------------------------------------------------------
TOTAL                       10,945      490   11,435    95.7%


## 📊 Complete Pipeline Flow → 10,225 Ground Truth Records

The ground truth dataset is built from **two parallel data pipelines** that converge at the end:

**PATH 1: Cochrane References → Included (label=1)**
- 119 PDFs → 17,368 references → matching → filtering → deduplication → **2,265 included records**

**PATH 2: PubMed Searches → Excluded (label=0)**  
- 40 searches → 9,767 excluded papers (not in any review) → filtering → **7,960 excluded records**

**Final Dataset**: 2,265 + 7,960 = **10,225 records** across 30 reviews (those with both included AND excluded papers)

The Sankey diagram below traces both paths from source to final ground truth.

In [205]:
# DATA QUALITY - COMPLETENESS METRICS

# Calculate completeness for references (using correct column names)
ref_completeness = {
    'Has Title': (references['title'].notna() & (references['title'] != '')).mean() * 100,
    'Has Year': (references['year'].notna()).mean() * 100,
    'Has DOI': (references['ref_doi'].notna() & (references['ref_doi'] != '')).mean() * 100 if 'ref_doi' in references.columns else 0,
    'Has Authors': (references['authors'].notna() & (references['authors'] != '')).mean() * 100,
}

# Calculate completeness for abstracts (using correct column names)
abstract_completeness = {
    'Has Abstract': (abstracts['abstract'].notna() & (abstracts['abstract'] != '')).mean() * 100,
    'Has PMID': (abstracts['pmid'].notna()).mean() * 100 if 'pmid' in abstracts.columns else 0,
    'Has Source': (abstracts['abstract_source'].notna() & (abstracts['abstract_source'] != '')).mean() * 100 if 'abstract_source' in abstracts.columns else 0,
}

# Combine
all_completeness = {**ref_completeness, **abstract_completeness}
completeness_df = pd.DataFrame({
    'Field': list(all_completeness.keys()),
    'Completeness': list(all_completeness.values())
}).sort_values('Completeness')

# Color based on completeness level
def get_quality_color(val):
    if val >= 90: return COLORS['success']
    elif val >= 70: return COLORS['warning']
    else: return COLORS['danger']

colors = [get_quality_color(v) for v in completeness_df['Completeness']]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=completeness_df['Field'],
    x=completeness_df['Completeness'],
    orientation='h',
    marker_color=colors,
    text=[f'{v:.1f}%' for v in completeness_df['Completeness']],
    textposition='outside',
    hovertemplate='%{y}: %{x:.1f}%<extra></extra>'
))

fig.update_layout(
    title='<b>Data Completeness by Field</b>',
    xaxis_title='Completeness (%)',
    yaxis_title='',
    height=350,
    template=template,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    xaxis=dict(range=[0, 110]),
    plot_bgcolor='rgba(0,0,0,0)'
)

# Add threshold lines
fig.add_vline(x=90, line_dash="dash", line_color=COLORS['success'], annotation_text="Excellent", annotation_position="top")
fig.add_vline(x=70, line_dash="dash", line_color=COLORS['warning'], annotation_text="Good", annotation_position="top")

fig.show()

In [213]:
# ══════════════════════════════════════════════════════════════════════════════
#  COMPLETE PIPELINE SANKEY: Both Paths → Ground Truth (10,225 records)
# ══════════════════════════════════════════════════════════════════════════════
# Shows the dual data flow:
#   PATH 1 (top): Cochrane PDFs → References → Matching → Dedup → label=1
#   PATH 2 (bottom): PubMed Searches → Excluded Papers → label=0
# Both paths converge into the final Ground Truth dataset

# ══════════════════════════════════════════════════════════════════════════════
# CALCULATE EXACT PIPELINE METRICS
# ══════════════════════════════════════════════════════════════════════════════

# ── PATH 1: Cochrane References (label=1) ──
total_refs = len(references)                                           # 17,368
matched_to_db = len(abstracts)                                         # 11,435
unmatched = total_refs - matched_to_db                                 # 5,933

has_good_abstract = (abstracts['abstract'].notna() & 
                     (abstracts['abstract'].str.len() > 50)).sum()     # 10,945
no_good_abstract = matched_to_db - has_good_abstract                   # 490

# Deduplication by (paper_key, review_doi)
def get_paper_key(row):
    if pd.notna(row.get('pmid')): return f"pmid:{int(row['pmid'])}"
    elif pd.notna(row.get('doi')): return f"doi:{row['doi']}"
    else: return f"title:{str(row.get('original_title', ''))[:100]}"

abstracts_with_abs = abstracts[abstracts['abstract'].notna() & (abstracts['abstract'].str.len() > 50)].copy()
abstracts_with_abs['paper_key'] = abstracts_with_abs.apply(get_paper_key, axis=1)
deduped_refs = abstracts_with_abs.drop_duplicates(subset=['paper_key', 'review_doi'])
after_dedup = len(deduped_refs)                                        # 10,624
collapsed_dups = has_good_abstract - after_dedup                       # 321

# Filter to 30 reviews with both classes (89 reviews excluded)
gt_reviews = set(ground_truth['review_doi'].unique())
n_reviews_final = len(gt_reviews)                                      # 30
n_reviews_excluded = 119 - n_reviews_final                             # 89

gt_cochrane = ground_truth[ground_truth['source'] == 'cochrane_ref']
final_label_1 = len(gt_cochrane)                                       # 2,265
dropped_cochrane = after_dedup - final_label_1                         # 8,359

# ── PATH 2: PubMed Excluded (label=0) ──
total_pubmed_excluded = len(pubmed_excluded)                           # 9,767
gt_pubmed = ground_truth[ground_truth['source'] == 'pubmed_search']
final_label_0 = len(gt_pubmed)                                         # 7,960
dropped_pubmed = total_pubmed_excluded - final_label_0                 # 1,807

# ── FINAL ──
total_ground_truth = len(ground_truth)                                 # 10,225

# ══════════════════════════════════════════════════════════════════════════════
# BUILD SANKEY DIAGRAM
# ══════════════════════════════════════════════════════════════════════════════

# Node labels with precise descriptions
nodes = [
    # PATH 1: Cochrane (top flow) - nodes 0-8
    '119 Cochrane PDFs',                                    # 0
    f'{total_refs:,} Extracted References',                 # 1
    f'{matched_to_db:,} Matched to Database',               # 2
    f'{unmatched:,} No Database Match',                     # 3
    f'{has_good_abstract:,} With Valid Abstract',           # 4
    f'{no_good_abstract:,} Missing Abstract',               # 5
    f'{after_dedup:,} Unique Papers',                       # 6
    f'{collapsed_dups:,} Duplicate Entries',                # 7
    f'{dropped_cochrane:,} Dropped (89 reviews)',           # 8  ← FIXED LABEL
    
    # PATH 2: PubMed (bottom flow) - nodes 9-11
    '40 PubMed Search Queries',                             # 9
    f'{total_pubmed_excluded:,} Not in Any Review',         # 10
    f'{dropped_pubmed:,} Dropped (89 reviews)',             # 11  ← FIXED LABEL
    
    # CONVERGENCE - nodes 12-14
    f'{final_label_1:,} Included',                          # 12
    f'{final_label_0:,} Excluded',                          # 13
    f'{total_ground_truth:,} Ground Truth',                 # 14
]

# Define flows (source_idx, target_idx, value)
flows_data = [
    # PATH 1: Cochrane reference pipeline
    (0, 1, total_refs),                          # PDFs → References
    (1, 2, matched_to_db),                       # References → Matched
    (1, 3, unmatched),                           # References → Unmatched
    (2, 4, has_good_abstract),                   # Matched → With Abstract
    (2, 5, no_good_abstract),                    # Matched → No Abstract
    (4, 6, after_dedup),                         # With Abstract → Unique
    (4, 7, collapsed_dups),                      # With Abstract → Duplicates
    (6, 12, final_label_1),                      # Unique → Included (30 reviews)
    (6, 8, dropped_cochrane),                    # Unique → Dropped (89 reviews)
    
    # PATH 2: PubMed excluded pipeline
    (9, 10, total_pubmed_excluded),              # Searches → Not in Reviews
    (10, 13, final_label_0),                     # Not in Reviews → Excluded (30 reviews)
    (10, 11, dropped_pubmed),                    # Not in Reviews → Dropped (89 reviews)
    
    # CONVERGENCE to Ground Truth
    (12, 14, final_label_1),                     # Included → GT
    (13, 14, final_label_0),                     # Excluded → GT
]

sources = [f[0] for f in flows_data]
targets = [f[1] for f in flows_data]
values = [f[2] for f in flows_data]

# Node colors
node_colors = [
    COLORS['primary'],    # 0: PDFs
    COLORS['info'],       # 1: References
    COLORS['success'],    # 2: Matched
    '#94A3B8',            # 3: Unmatched (gray - dropped)
    COLORS['success'],    # 4: With Abstract
    '#94A3B8',            # 5: No Abstract (gray - dropped)
    COLORS['success'],    # 6: Unique
    '#CBD5E1',            # 7: Duplicates (light gray - dropped)
    '#94A3B8',            # 8: Dropped cochrane (gray)
    COLORS['secondary'],  # 9: PubMed Searches
    COLORS['danger'],     # 10: Not in Reviews
    '#94A3B8',            # 11: Dropped pubmed (gray)
    COLORS['success'],    # 12: Included (label=1)
    COLORS['danger'],     # 13: Excluded (label=0)
    COLORS['primary'],    # 14: Ground Truth
]

# Link colors
link_colors = [
    'rgba(59, 130, 246, 0.4)',   # PDFs → Refs (blue)
    'rgba(16, 185, 129, 0.4)',   # Refs → Matched (green)
    'rgba(148, 163, 184, 0.3)',  # Refs → Unmatched (gray)
    'rgba(16, 185, 129, 0.4)',   # Matched → With Abstract (green)
    'rgba(148, 163, 184, 0.3)',  # Matched → No Abstract (gray)
    'rgba(16, 185, 129, 0.4)',   # With Abs → Unique (green)
    'rgba(200, 200, 200, 0.3)',  # With Abs → Duplicates (gray)
    'rgba(16, 185, 129, 0.5)',   # Unique → Included (green)
    'rgba(148, 163, 184, 0.3)',  # Unique → Dropped (gray)
    'rgba(124, 58, 237, 0.4)',   # Searches → Not in Reviews (purple)
    'rgba(239, 68, 68, 0.5)',    # Not in Reviews → Excluded (red)
    'rgba(148, 163, 184, 0.3)',  # Not in Reviews → Dropped (gray)
    'rgba(16, 185, 129, 0.5)',   # Included → GT (green)
    'rgba(239, 68, 68, 0.5)',    # Excluded → GT (red)
]

# Node positions - cleaner layout with more separation
node_x = [
    0.001, # 0: PDFs
    0.13,  # 1: References
    0.26,  # 2: Matched
    0.26,  # 3: Unmatched
    0.39,  # 4: With Abstract
    0.39,  # 5: No Abstract
    0.52,  # 6: Unique
    0.52,  # 7: Duplicates
    0.65,  # 8: Dropped (cochrane)
    0.001, # 9: PubMed Searches
    0.26,  # 10: Not in Reviews
    0.52,  # 11: Dropped (pubmed)
    0.78,  # 12: Included
    0.78,  # 13: Excluded
    0.97,  # 14: Ground Truth
]

node_y = [
    0.20,  # 0: PDFs (top path)
    0.20,  # 1: References
    0.15,  # 2: Matched
    0.45,  # 3: Unmatched (below, dropped)
    0.12,  # 4: With Abstract
    0.38,  # 5: No Abstract (below, dropped)
    0.10,  # 6: Unique
    0.32,  # 7: Duplicates (below, dropped)
    0.28,  # 8: Dropped cochrane
    0.80,  # 9: PubMed Searches (bottom path)
    0.78,  # 10: Not in Reviews
    0.92,  # 11: Dropped pubmed
    0.18,  # 12: Included
    0.72,  # 13: Excluded
    0.45,  # 14: Ground Truth (center)
]

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=25,
        thickness=20,
        line=dict(color='white', width=1),
        label=nodes,
        color=node_colors,
        x=node_x,
        y=node_y,
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title=dict(
        text='<b>Ground Truth Construction Pipeline</b>',
        x=0.5,
        y=0.98,
        font=dict(size=18)
    ),
    height=700,
    font=dict(family='Segoe UI, sans-serif', size=10),
    margin=dict(l=10, r=10, t=80, b=50),
    annotations=[
        # Top path label - left aligned, above the diagram
        dict(x=0.0, y=1.06, xref='paper', yref='paper', showarrow=False,
             text='<b>PATH 1:</b> Cochrane References → label=1 (Included)', 
             font=dict(size=11, color=COLORS['success']), xanchor='left'),
        # Bottom path label - left aligned, below the diagram
        dict(x=0.0, y=-0.01, xref='paper', yref='paper', showarrow=False,
             text='<b>PATH 2:</b> PubMed Search Results → label=0 (Excluded)', 
             font=dict(size=11, color=COLORS['danger']), xanchor='left'),
        # Explanation of "89 reviews" - positioned in the middle
        dict(x=0.65, y=0.50, xref='paper', yref='paper', showarrow=False,
             text='<i>89 reviews lack both classes<br>→ dropped from ground truth</i>', 
             font=dict(size=9, color='#64748B'), align='center'),
    ]
)

fig.show()

# Save to HTML
import os
vis_dir = '../Visualisations'
os.makedirs(vis_dir, exist_ok=True)
html_path = f'{vis_dir}/ground_truth_pipeline.html'
fig.write_html(html_path, include_plotlyjs='cdn')
print(f"✓ Saved visualization to: {html_path}")

# ══════════════════════════════════════════════════════════════════════════════
# PIPELINE SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*80)
print("GROUND TRUTH PIPELINE SUMMARY")
print("="*80)

print(f"\n🔵 PATH 1: COCHRANE REFERENCES → INCLUDED (label=1)")
print(f"   ├─ 119 PDFs → {total_refs:,} references extracted")
print(f"   ├─ {matched_to_db:,} matched to CrossRef/PubMed ({matched_to_db/total_refs*100:.1f}%)")
print(f"   │   └─ {unmatched:,} no database match (dropped)")
print(f"   ├─ {has_good_abstract:,} with valid abstract >50 chars")
print(f"   │   └─ {no_good_abstract:,} missing/short abstract (dropped)")
print(f"   ├─ {after_dedup:,} unique papers after deduplication")
print(f"   │   └─ {collapsed_dups:,} duplicate entries (dropped)")
print(f"   ├─ {final_label_1:,} papers from {n_reviews_final} reviews with both classes")
print(f"   │   └─ {dropped_cochrane:,} papers from {n_reviews_excluded} reviews without negatives (dropped)")
print(f"   └─ ✅ {final_label_1:,} records (label=1)")

print(f"\n🔴 PATH 2: PUBMED SEARCHES → EXCLUDED (label=0)")
print(f"   ├─ 40 search queries executed")
print(f"   ├─ {total_pubmed_excluded:,} papers not cited in any Cochrane review")
print(f"   ├─ {final_label_0:,} papers from {n_reviews_final} reviews with both classes")
print(f"   │   └─ {dropped_pubmed:,} papers from {n_reviews_excluded} reviews without positives (dropped)")
print(f"   └─ ✅ {final_label_0:,} records (label=0)")

print(f"\n📊 FINAL GROUND TRUTH DATASET")
print(f"   ├─ Reviews: {n_reviews_final} (of 119 total, {n_reviews_excluded} excluded)")
print(f"   ├─ Included (label=1): {final_label_1:,} ({final_label_1/total_ground_truth*100:.1f}%)")
print(f"   ├─ Excluded (label=0): {final_label_0:,} ({final_label_0/total_ground_truth*100:.1f}%)")
print(f"   └─ Total records: {total_ground_truth:,}")
print("="*80)

✓ Saved visualization to: ../Visualisations/ground_truth_pipeline.html

GROUND TRUTH PIPELINE SUMMARY

🔵 PATH 1: COCHRANE REFERENCES → INCLUDED (label=1)
   ├─ 119 PDFs → 17,368 references extracted
   ├─ 11,435 matched to CrossRef/PubMed (65.8%)
   │   └─ 5,933 no database match (dropped)
   ├─ 10,945 with valid abstract >50 chars
   │   └─ 490 missing/short abstract (dropped)
   ├─ 10,624 unique papers after deduplication
   │   └─ 321 duplicate entries (dropped)
   ├─ 2,265 papers from 30 reviews with both classes
   │   └─ 8,359 papers from 89 reviews without negatives (dropped)
   └─ ✅ 2,265 records (label=1)

🔴 PATH 2: PUBMED SEARCHES → EXCLUDED (label=0)
   ├─ 40 search queries executed
   ├─ 9,767 papers not cited in any Cochrane review
   ├─ 7,960 papers from 30 reviews with both classes
   │   └─ 1,807 papers from 89 reviews without positives (dropped)
   └─ ✅ 7,960 records (label=0)

📊 FINAL GROUND TRUTH DATASET
   ├─ Reviews: 30 (of 119 total, 89 excluded)
   ├─ Included (l

## 🔎 Interactive Data Explorer

Top reviews by total references extracted.

In [207]:
# INTERACTIVE REVIEW TABLE

# Build summary per review (using review_doi as the key)
review_summary = references.groupby('review_doi').agg({
    'study_id': 'count',
}).rename(columns={'study_id': 'total_refs'})

# Add category breakdowns
for cat in ['included', 'excluded', 'additional', 'awaiting', 'ongoing']:
    cat_counts = references[references['category'] == cat].groupby('review_doi').size()
    review_summary[cat] = cat_counts.reindex(review_summary.index, fill_value=0)

# Merge with metadata (reviews table uses 'doi' and 'year')
review_summary = review_summary.reset_index()
reviews_info = reviews[['doi', 'title', 'year']].copy()
reviews_info.columns = ['review_doi', 'review_title', 'review_year']
review_summary = review_summary.merge(reviews_info, on='review_doi', how='left')

# Sort by total references
review_summary = review_summary.sort_values('total_refs', ascending=False)

# Create interactive table
fig = go.Figure(data=[go.Table(
    columnwidth=[50, 200, 40, 50, 50, 50, 50, 50, 50],
    header=dict(
        values=['<b>Review DOI</b>', '<b>Title</b>', '<b>Year</b>', '<b>Total</b>', 
                '<b>Included</b>', '<b>Excluded</b>', '<b>Additional</b>', '<b>Awaiting</b>', '<b>Ongoing</b>'],
        fill_color=COLORS['primary'],
        font=dict(color='white', size=11),
        align='left',
        height=35
    ),
    cells=dict(
        values=[
            review_summary['review_doi'].head(20),
            review_summary['review_title'].head(20).apply(lambda x: x[:60] + '...' if isinstance(x, str) and len(x) > 60 else x),
            review_summary['review_year'].head(20),
            review_summary['total_refs'].head(20),
            review_summary['included'].head(20),
            review_summary['excluded'].head(20),
            review_summary['additional'].head(20),
            review_summary['awaiting'].head(20),
            review_summary['ongoing'].head(20)
        ],
        fill_color=[['white', '#f8fafc'] * 10],
        font=dict(size=10),
        align='left',
        height=28
    )
)])

fig.update_layout(
    title='<b>Top 20 Reviews by References Extracted</b>',
    height=700,
    font_family='Segoe UI, sans-serif',
    title_font_size=18,
    title_x=0.5,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()

## 🎯 Ground Truth Dataset Visualization

Interactive visualization of the **10,225** paper-review pairs used for LLM screening evaluation.

In [214]:
# ══════════════════════════════════════════════════════════════════════════════
#  GROUND TRUTH DATASET - INTERACTIVE VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════
# Professional dashboard for presentation showing:
#   - Dataset composition (sunburst)
#   - Per-review breakdown (horizontal bar)
#   - Class balance analysis

from plotly.subplots import make_subplots

# ── CALCULATE METRICS ──
gt = ground_truth.copy()

# Per-review statistics
review_stats = gt.groupby('review_doi').agg({
    'label': ['sum', 'count'],
    'review_title': 'first'
}).reset_index()
review_stats.columns = ['review_doi', 'included', 'total', 'review_title']
review_stats['excluded'] = review_stats['total'] - review_stats['included']
review_stats['inclusion_rate'] = review_stats['included'] / review_stats['total']
review_stats = review_stats.sort_values('total', ascending=True)

# Extract short review ID
review_stats['review_id'] = review_stats['review_doi'].str.extract(r'(CD\d+)')

# Truncate titles for display
review_stats['short_title'] = review_stats['review_title'].str[:60].str.replace(
    r'\s+\(Review\).*', '', regex=True
).str.strip() + '...'

# Overall stats
total_papers = len(gt)
n_included = gt['label'].sum()
n_excluded = total_papers - n_included
n_reviews = len(review_stats)
avg_inclusion_rate = review_stats['inclusion_rate'].mean()

# ══════════════════════════════════════════════════════════════════════════════
#  CREATE MULTI-PANEL FIGURE
# ══════════════════════════════════════════════════════════════════════════════

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}]],
    column_widths=[0.35, 0.65],
    horizontal_spacing=0.08,
    subplot_titles=('Dataset Composition', 'Papers per Systematic Review')
)

# ── PANEL 1: SUNBURST ──
# Create hierarchical data for sunburst
sunburst_labels = [
    'Ground Truth<br>Dataset',  # Root
    f'Included<br>(label=1)',   # Level 1
    f'Excluded<br>(label=0)',   # Level 1
]
sunburst_parents = ['', 'Ground Truth<br>Dataset', 'Ground Truth<br>Dataset']
sunburst_values = [total_papers, n_included, n_excluded]
sunburst_colors = ['#3B82F6', COLORS['success'], COLORS['danger']]

fig.add_trace(
    go.Sunburst(
        labels=sunburst_labels,
        parents=sunburst_parents,
        values=sunburst_values,
        branchvalues='total',
        marker=dict(colors=sunburst_colors, line=dict(color='white', width=2)),
        texttemplate='%{label}<br><b>%{value:,}</b><br>(%{percentRoot:.1%})',
        textfont=dict(size=12, color='white'),
        hovertemplate='<b>%{label}</b><br>Papers: %{value:,}<br>Share: %{percentRoot:.1%}<extra></extra>',
        insidetextorientation='horizontal',
    ),
    row=1, col=1
)

# ── PANEL 2: HORIZONTAL STACKED BAR ──
fig.add_trace(
    go.Bar(
        y=review_stats['review_id'],
        x=review_stats['included'],
        name='Included (label=1)',
        orientation='h',
        marker=dict(color=COLORS['success'], line=dict(color='white', width=0.5)),
        text=review_stats['included'],
        textposition='inside',
        textfont=dict(size=9, color='white'),
        hovertemplate='<b>%{y}</b><br>Included: %{x:,} papers<extra></extra>',
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        y=review_stats['review_id'],
        x=review_stats['excluded'],
        name='Excluded (label=0)',
        orientation='h',
        marker=dict(color=COLORS['danger'], line=dict(color='white', width=0.5)),
        text=review_stats['excluded'],
        textposition='inside',
        textfont=dict(size=9, color='white'),
        hovertemplate='<b>%{y}</b><br>Excluded: %{x:,} papers<extra></extra>',
    ),
    row=1, col=2
)

# ── LAYOUT ──
fig.update_layout(
    title=dict(
        text=f'<b>Ground Truth Validation Dataset</b><br>'
             f'<span style="font-size:13px;color:#64748B">'
             f'{total_papers:,} paper-review pairs across {n_reviews} systematic reviews | '
             f'Inclusion rate: {avg_inclusion_rate:.1%}</span>',
        x=0.5,
        y=0.97,
        font=dict(size=18)
    ),
    height=750,
    barmode='stack',
    font=dict(family='Segoe UI, sans-serif', size=11),
    margin=dict(l=10, r=10, t=100, b=60),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.06,
        xanchor='center',
        x=0.65,
        font=dict(size=11),
        bgcolor='rgba(255,255,255,0.8)'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

# Style x-axis
fig.update_xaxes(
    title_text='Number of Papers',
    title_font=dict(size=11),
    gridcolor='#E5E7EB',
    gridwidth=1,
    zeroline=False,
    row=1, col=2
)

# Style y-axis
fig.update_yaxes(
    title_text='',
    tickfont=dict(size=9),
    row=1, col=2
)

# Update subplot titles
fig.update_annotations(font=dict(size=13))

fig.show()

# Save to HTML
html_path = f'{vis_dir}/ground_truth_overview.html'
fig.write_html(html_path, include_plotlyjs='cdn')
print(f"✓ Saved visualization to: {html_path}")

✓ Saved visualization to: ../Visualisations/ground_truth_overview.html


In [226]:
# ══════════════════════════════════════════════════════════════════════════════
#  INTERACTIVE DRILL-DOWN VISUALIZATION WITH PAPER DETAILS
# ══════════════════════════════════════════════════════════════════════════════
# Click on any review bar to see included/excluded papers with abstracts
# Back button returns to the overview

import json
import html as html_lib

# Sort by total papers descending for this view
review_data = review_stats.sort_values('total', ascending=False).reset_index(drop=True)

# Add review abstract from ground_truth (get first occurrence for each review)
review_abstracts = ground_truth.groupby('review_doi')['review_abstract'].first().to_dict()
review_data['review_abstract'] = review_data['review_doi'].map(review_abstracts)

# Create custom hover text with full details (black text)
review_data['hover_text'] = (
    '<b>' + review_data['review_id'] + '</b><br>' +
    '<i>' + review_data['review_title'].str[:80] + '...</i><br><br>' +
    'Accepted: ' + review_data['included'].astype(str) + ' papers<br>' +
    'Rejected: ' + review_data['excluded'].astype(str) + ' papers<br>' +
    'Total: ' + review_data['total'].astype(str) + ' papers<br>' +
    'Inclusion rate: ' + (review_data['inclusion_rate'] * 100).round(1).astype(str) + '%'
)

fig2 = go.Figure()

# Add included bars
fig2.add_trace(go.Bar(
    x=review_data['review_id'],
    y=review_data['included'],
    name='Accepted (label=1)',
    marker=dict(
        color=COLORS['success'],
        line=dict(color='white', width=1)
    ),
    hovertext=review_data['hover_text'],
    hoverinfo='text',
    text=review_data['included'],
    textposition='inside',
    textfont=dict(size=10, color='white'),
))

# Add excluded bars
fig2.add_trace(go.Bar(
    x=review_data['review_id'],
    y=review_data['excluded'],
    name='Rejected (label=0)',
    marker=dict(
        color=COLORS['danger'],
        line=dict(color='white', width=1)
    ),
    hovertext=review_data['hover_text'],
    hoverinfo='text',
    text=review_data['excluded'],
    textposition='inside',
    textfont=dict(size=10, color='white'),
))

# Add inclusion rate line on secondary axis
fig2.add_trace(go.Scatter(
    x=review_data['review_id'],
    y=review_data['inclusion_rate'] * 100,
    name='Inclusion Rate (%)',
    mode='lines+markers',
    line=dict(color='#F59E0B', width=3),
    marker=dict(size=8, color='#F59E0B', line=dict(color='white', width=1)),
    yaxis='y2',
    hovertemplate='<b>%{x}</b><br>Inclusion rate: %{y:.1f}%<extra></extra>',
))

fig2.update_layout(
    title=dict(
        text='<b>Ground Truth Dataset: Per-Review Breakdown</b><br>'
             '<span style="font-size:12px;color:#64748B">'
             'Sorted by total papers (descending) | Click on a bar to view papers</span>',
        x=0.5,
        y=0.97,
        font=dict(size=16)
    ),
    barmode='stack',
    height=500,
    font=dict(family='Segoe UI, sans-serif', size=11),
    margin=dict(l=60, r=60, t=90, b=80),
    xaxis=dict(
        title='Systematic Review',
        tickangle=45,
        tickfont=dict(size=9),
        gridcolor='#E5E7EB',
    ),
    yaxis=dict(
        title='Number of Papers',
        gridcolor='#E5E7EB',
        zeroline=False,
    ),
    yaxis2=dict(
        title='Inclusion Rate (%)',
        overlaying='y',
        side='right',
        range=[0, 30],
        ticksuffix='%',
        gridcolor='rgba(245, 158, 11, 0.1)',
        zeroline=False,
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
        font=dict(size=10),
        bgcolor='rgba(255,255,255,0.9)'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='#CBD5E1',
        font=dict(size=11, family='Segoe UI, sans-serif', color='#1E293B')
    ),
)

fig2.show()

# ══════════════════════════════════════════════════════════════════════════════
#  BUILD INTERACTIVE HTML WITH DRILL-DOWN
# ══════════════════════════════════════════════════════════════════════════════

def escape_js_string(s):
    """Escape string for JavaScript."""
    if pd.isna(s):
        return ''
    return str(s).replace('\\', '\\\\').replace("'", "\\'").replace('"', '\\"').replace('\n', '\\n').replace('\r', '')

# Prepare paper data for each review
papers_by_review = {}
for review_doi in review_data['review_doi']:
    review_papers = ground_truth[ground_truth['review_doi'] == review_doi].copy()
    papers_list = []
    for _, paper in review_papers.iterrows():
        papers_list.append({
            'paper_id': escape_js_string(paper['paper_id']),
            'title': escape_js_string(paper['paper_title'][:150] + '...' if len(str(paper['paper_title'])) > 150 else paper['paper_title']),
            'abstract': escape_js_string(paper['paper_abstract'][:500] + '...' if len(str(paper['paper_abstract'])) > 500 else paper['paper_abstract']),
            'full_abstract': escape_js_string(paper['paper_abstract']),
            'label': int(paper['label']),
        })
    papers_by_review[review_doi] = papers_list

# Get main chart HTML
main_chart_html = fig2.to_html(include_plotlyjs='cdn', full_html=False, div_id='main-chart')

# Build review info for JS
review_info_js = {}
for _, row in review_data.iterrows():
    review_info_js[row['review_id']] = {
        'doi': row['review_doi'],
        'title': escape_js_string(row['review_title']),
        'abstract': escape_js_string(row['review_abstract']),
        'included': int(row['included']),
        'excluded': int(row['excluded']),
        'total': int(row['total']),
        'inclusion_rate': round(row['inclusion_rate'] * 100, 1)
    }

# Convert papers data to JSON
papers_json = json.dumps(papers_by_review)
review_info_json = json.dumps(review_info_js)

# Build interactive HTML
interactive_html = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ground Truth Dataset Explorer</title>
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{ 
            font-family: 'Segoe UI', system-ui, sans-serif; 
            background: #F8FAFC; 
            color: #1E293B;
            line-height: 1.6;
        }}
        .container {{ max-width: 1400px; margin: 0 auto; padding: 20px; }}
        
        /* Views */
        #overview-view, #detail-view {{ display: none; }}
        #overview-view.active, #detail-view.active {{ display: block; }}
        
        /* Back button */
        .back-btn {{
            display: inline-flex;
            align-items: center;
            gap: 8px;
            padding: 10px 20px;
            background: #3B82F6;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 14px;
            font-weight: 500;
            transition: background 0.2s;
            margin-bottom: 20px;
        }}
        .back-btn:hover {{ background: #2563EB; }}
        .back-btn svg {{ width: 16px; height: 16px; }}
        
        /* Review detail header */
        .review-header {{
            background: white;
            border-radius: 12px;
            padding: 24px;
            margin-bottom: 20px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}
        .review-header h2 {{ font-size: 18px; margin-bottom: 8px; color: #1E293B; }}
        .review-header .review-title {{ color: #64748B; font-size: 14px; margin-bottom: 12px; }}
        .review-header .review-abstract {{
            font-size: 13px;
            color: #475569;
            line-height: 1.6;
            margin-bottom: 16px;
            padding: 12px 16px;
            background: #F8FAFC;
            border-radius: 8px;
            border-left: 3px solid #3B82F6;
            max-height: 150px;
            overflow-y: auto;
        }}
        .stats-row {{
            display: flex;
            gap: 24px;
            flex-wrap: wrap;
        }}
        .stat-item {{
            display: flex;
            flex-direction: column;
        }}
        .stat-label {{ font-size: 12px; color: #64748B; text-transform: uppercase; letter-spacing: 0.5px; }}
        .stat-value {{ font-size: 24px; font-weight: 600; }}
        .stat-value.included {{ color: #10B981; }}
        .stat-value.excluded {{ color: #EF4444; }}
        .stat-value.total {{ color: #3B82F6; }}
        
        /* Filter tabs */
        .filter-tabs {{
            display: flex;
            gap: 8px;
            margin-bottom: 16px;
        }}
        .filter-tab {{
            padding: 8px 16px;
            border: 2px solid #E2E8F0;
            background: white;
            border-radius: 8px;
            cursor: pointer;
            font-size: 13px;
            font-weight: 500;
            transition: all 0.2s;
        }}
        .filter-tab:hover {{ border-color: #CBD5E1; }}
        .filter-tab.active {{ border-color: #3B82F6; background: #EFF6FF; color: #3B82F6; }}
        .filter-tab.included.active {{ border-color: #10B981; background: #ECFDF5; color: #059669; }}
        .filter-tab.excluded.active {{ border-color: #EF4444; background: #FEF2F2; color: #DC2626; }}
        
        /* Papers table */
        .papers-container {{
            background: white;
            border-radius: 12px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            overflow: hidden;
        }}
        .papers-table {{
            width: 100%;
            border-collapse: collapse;
        }}
        .papers-table th {{
            background: #F8FAFC;
            padding: 12px 16px;
            text-align: left;
            font-size: 12px;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            color: #64748B;
            border-bottom: 1px solid #E2E8F0;
        }}
        .papers-table td {{
            padding: 16px;
            border-bottom: 1px solid #F1F5F9;
            vertical-align: top;
        }}
        .papers-table tr:hover {{ background: #F8FAFC; }}
        .paper-title {{ font-weight: 500; margin-bottom: 8px; color: #1E293B; }}
        .paper-abstract {{ 
            font-size: 13px; 
            color: #64748B; 
            line-height: 1.5;
            cursor: pointer;
        }}
        .paper-abstract:hover {{ color: #475569; }}
        .paper-id {{ font-size: 12px; color: #94A3B8; font-family: monospace; }}
        .label-badge {{
            display: inline-block;
            padding: 4px 10px;
            border-radius: 12px;
            font-size: 11px;
            font-weight: 600;
            text-transform: uppercase;
        }}
        .label-badge.included {{ background: #ECFDF5; color: #059669; }}
        .label-badge.excluded {{ background: #FEF2F2; color: #DC2626; }}
        
        /* Modal for full abstract */
        .modal {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            background: rgba(0,0,0,0.5);
            z-index: 1000;
            align-items: center;
            justify-content: center;
        }}
        .modal.active {{ display: flex; }}
        .modal-content {{
            background: white;
            border-radius: 12px;
            max-width: 800px;
            max-height: 80vh;
            overflow-y: auto;
            padding: 24px;
            margin: 20px;
            box-shadow: 0 20px 40px rgba(0,0,0,0.2);
        }}
        .modal-header {{
            display: flex;
            justify-content: space-between;
            align-items: flex-start;
            margin-bottom: 16px;
        }}
        .modal-header h3 {{ font-size: 16px; color: #1E293B; flex: 1; }}
        .modal-close {{
            width: 32px;
            height: 32px;
            border: none;
            background: #F1F5F9;
            border-radius: 8px;
            cursor: pointer;
            font-size: 18px;
            color: #64748B;
        }}
        .modal-close:hover {{ background: #E2E8F0; }}
        .modal-body {{ font-size: 14px; color: #475569; line-height: 1.7; }}
        
        /* Pagination */
        .pagination {{
            display: flex;
            justify-content: center;
            gap: 8px;
            padding: 16px;
            border-top: 1px solid #E2E8F0;
        }}
        .page-btn {{
            padding: 8px 12px;
            border: 1px solid #E2E8F0;
            background: white;
            border-radius: 6px;
            cursor: pointer;
            font-size: 13px;
        }}
        .page-btn:hover {{ background: #F8FAFC; }}
        .page-btn.active {{ background: #3B82F6; color: white; border-color: #3B82F6; }}
        .page-btn:disabled {{ opacity: 0.5; cursor: not-allowed; }}
    </style>
</head>
<body>
    <div class="container">
        <!-- Overview View -->
        <div id="overview-view" class="active">
            {main_chart_html}
        </div>
        
        <!-- Detail View -->
        <div id="detail-view">
            <button class="back-btn" onclick="showOverview()">
                <svg fill="none" stroke="currentColor" viewBox="0 0 24 24">
                    <path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M10 19l-7-7m0 0l7-7m-7 7h18"/>
                </svg>
                Back to Overview
            </button>
            
            <div class="review-header">
                <h2 id="detail-review-id"></h2>
                <div class="review-title" id="detail-review-title"></div>
                <div class="review-abstract" id="detail-review-abstract"></div>
                <div class="stats-row">
                    <div class="stat-item">
                        <span class="stat-label">Accepted</span>
                        <span class="stat-value included" id="detail-included"></span>
                    </div>
                    <div class="stat-item">
                        <span class="stat-label">Rejected</span>
                        <span class="stat-value excluded" id="detail-excluded"></span>
                    </div>
                    <div class="stat-item">
                        <span class="stat-label">Total Papers</span>
                        <span class="stat-value total" id="detail-total"></span>
                    </div>
                    <div class="stat-item">
                        <span class="stat-label">Inclusion Rate</span>
                        <span class="stat-value" id="detail-rate"></span>
                    </div>
                </div>
            </div>
            
            <div class="filter-tabs">
                <button class="filter-tab active" data-filter="all" onclick="setFilter('all', this)">All Papers</button>
                <button class="filter-tab included" data-filter="1" onclick="setFilter(1, this)">Accepted Only</button>
                <button class="filter-tab excluded" data-filter="0" onclick="setFilter(0, this)">Rejected Only</button>
            </div>
            
            <div class="papers-container">
                <table class="papers-table">
                    <thead>
                        <tr>
                            <th style="width:50px">#</th>
                            <th style="width:100px">Label</th>
                            <th style="width:120px">Paper ID</th>
                            <th>Title & Abstract</th>
                        </tr>
                    </thead>
                    <tbody id="papers-tbody"></tbody>
                </table>
                <div class="pagination" id="pagination"></div>
            </div>
        </div>
    </div>
    
    <!-- Modal for full abstract -->
    <div class="modal" id="abstract-modal">
        <div class="modal-content">
            <div class="modal-header">
                <h3 id="modal-title"></h3>
                <button class="modal-close" onclick="closeModal()">&times;</button>
            </div>
            <div class="modal-body" id="modal-body"></div>
        </div>
    </div>
    
    <script>
        // Data
        const papersData = {papers_json};
        const reviewInfo = {review_info_json};
        
        let currentReview = null;
        let currentFilter = 'all';
        let currentPage = 1;
        const pageSize = 20;
        
        // Attach click handler to chart
        document.getElementById('main-chart').on('plotly_click', function(data) {{
            if (data.points && data.points[0]) {{
                const reviewId = data.points[0].x;
                showReviewDetail(reviewId);
            }}
        }});
        
        function showOverview() {{
            document.getElementById('overview-view').classList.add('active');
            document.getElementById('detail-view').classList.remove('active');
            window.scrollTo(0, 0);
        }}
        
        function showReviewDetail(reviewId) {{
            const info = reviewInfo[reviewId];
            if (!info) return;
            
            currentReview = reviewId;
            currentFilter = 'all';
            currentPage = 1;
            
            // Update header
            document.getElementById('detail-review-id').textContent = reviewId;
            document.getElementById('detail-review-title').textContent = info.title;
            document.getElementById('detail-review-abstract').textContent = info.abstract;
            document.getElementById('detail-included').textContent = info.included;
            document.getElementById('detail-excluded').textContent = info.excluded;
            document.getElementById('detail-total').textContent = info.total;
            document.getElementById('detail-rate').textContent = info.inclusion_rate + '%';
            
            // Reset filter tabs
            document.querySelectorAll('.filter-tab').forEach(tab => tab.classList.remove('active'));
            document.querySelector('.filter-tab[data-filter="all"]').classList.add('active');
            
            // Switch views
            document.getElementById('overview-view').classList.remove('active');
            document.getElementById('detail-view').classList.add('active');
            
            renderPapers();
            window.scrollTo(0, 0);
        }}
        
        function setFilter(filter, btn) {{
            currentFilter = filter;
            currentPage = 1;
            document.querySelectorAll('.filter-tab').forEach(tab => tab.classList.remove('active'));
            btn.classList.add('active');
            renderPapers();
        }}
        
        function getFilteredPapers() {{
            const info = reviewInfo[currentReview];
            const papers = papersData[info.doi] || [];
            if (currentFilter === 'all') return papers;
            return papers.filter(p => p.label === currentFilter);
        }}
        
        function renderPapers() {{
            const papers = getFilteredPapers();
            const totalPages = Math.ceil(papers.length / pageSize);
            const start = (currentPage - 1) * pageSize;
            const pagePapers = papers.slice(start, start + pageSize);
            
            const tbody = document.getElementById('papers-tbody');
            tbody.innerHTML = pagePapers.map((p, i) => `
                <tr>
                    <td style="color:#94A3B8">${{start + i + 1}}</td>
                    <td><span class="label-badge ${{p.label === 1 ? 'included' : 'excluded'}}">${{p.label === 1 ? 'Accepted' : 'Rejected'}}</span></td>
                    <td><span class="paper-id">${{p.paper_id}}</span></td>
                    <td>
                        <div class="paper-title">${{p.title}}</div>
                        <div class="paper-abstract" onclick="showFullAbstract('${{p.title.replace(/'/g, "\\\\'")}}',' ${{p.full_abstract.replace(/'/g, "\\\\'")}}')">${{p.abstract}}</div>
                    </td>
                </tr>
            `).join('');
            
            // Render pagination
            const pagination = document.getElementById('pagination');
            if (totalPages <= 1) {{
                pagination.innerHTML = '';
                return;
            }}
            
            let paginationHtml = `<button class="page-btn" onclick="goToPage(1)" ${{currentPage === 1 ? 'disabled' : ''}}>First</button>`;
            paginationHtml += `<button class="page-btn" onclick="goToPage(${{currentPage - 1}})" ${{currentPage === 1 ? 'disabled' : ''}}>&lt;</button>`;
            
            const startPage = Math.max(1, currentPage - 2);
            const endPage = Math.min(totalPages, currentPage + 2);
            
            for (let i = startPage; i <= endPage; i++) {{
                paginationHtml += `<button class="page-btn ${{i === currentPage ? 'active' : ''}}" onclick="goToPage(${{i}})">${{i}}</button>`;
            }}
            
            paginationHtml += `<button class="page-btn" onclick="goToPage(${{currentPage + 1}})" ${{currentPage === totalPages ? 'disabled' : ''}}>&gt;</button>`;
            paginationHtml += `<button class="page-btn" onclick="goToPage(${{totalPages}})" ${{currentPage === totalPages ? 'disabled' : ''}}>Last</button>`;
            paginationHtml += `<span style="margin-left:16px;color:#64748B;font-size:13px">Page ${{currentPage}} of ${{totalPages}} (${{papers.length}} papers)</span>`;
            
            pagination.innerHTML = paginationHtml;
        }}
        
        function goToPage(page) {{
            const papers = getFilteredPapers();
            const totalPages = Math.ceil(papers.length / pageSize);
            if (page < 1 || page > totalPages) return;
            currentPage = page;
            renderPapers();
            document.querySelector('.papers-container').scrollIntoView({{ behavior: 'smooth' }});
        }}
        
        function showFullAbstract(title, abstract) {{
            document.getElementById('modal-title').textContent = title;
            document.getElementById('modal-body').textContent = abstract;
            document.getElementById('abstract-modal').classList.add('active');
        }}
        
        function closeModal() {{
            document.getElementById('abstract-modal').classList.remove('active');
        }}
        
        // Close modal on backdrop click
        document.getElementById('abstract-modal').addEventListener('click', function(e) {{
            if (e.target === this) closeModal();
        }});
        
        // Escape key closes modal
        document.addEventListener('keydown', function(e) {{
            if (e.key === 'Escape') closeModal();
        }});
    </script>
</body>
</html>
'''

# Save interactive HTML
html_path2 = f'{vis_dir}/ground_truth_explorer.html'
with open(html_path2, 'w', encoding='utf-8') as f:
    f.write(interactive_html)

print(f"✓ Saved interactive visualization to: {html_path2}")
print(f"  → Click any bar to drill down into papers")
print(f"  → Filter by Included/Excluded")
print(f"  → Click abstract to view full text")
print(f"  → Back button returns to overview")

✓ Saved interactive visualization to: ../Visualisations/ground_truth_explorer.html
  → Click any bar to drill down into papers
  → Filter by Included/Excluded
  → Click abstract to view full text
  → Back button returns to overview


In [216]:
# ══════════════════════════════════════════════════════════════════════════════
#  PRESENTATION-READY: TREEMAP + KEY STATS
# ══════════════════════════════════════════════════════════════════════════════

# Build hierarchical data for treemap: Dataset → Review → Label
treemap_data = []

for _, row in review_stats.iterrows():
    # Add included papers for this review
    if row['included'] > 0:
        treemap_data.append({
            'review_id': row['review_id'],
            'review_title': row['review_title'][:60] + '...',
            'label': 'Included',
            'count': row['included'],
            'parent': row['review_id'],
            'color': COLORS['success']
        })
    # Add excluded papers for this review
    if row['excluded'] > 0:
        treemap_data.append({
            'review_id': row['review_id'],
            'review_title': row['review_title'][:60] + '...',
            'label': 'Excluded',  
            'count': row['excluded'],
            'parent': row['review_id'],
            'color': COLORS['danger']
        })

treemap_df = pd.DataFrame(treemap_data)

# Create labels and parents for treemap
ids = []
labels = []
parents = []
values = []
colors = []

# Root
ids.append('Dataset')
labels.append(f'Ground Truth<br><b>{total_papers:,}</b> papers')
parents.append('')
values.append(total_papers)
colors.append('#3B82F6')

# Reviews level
for _, row in review_stats.sort_values('total', ascending=False).iterrows():
    ids.append(row['review_id'])
    labels.append(f"<b>{row['review_id']}</b><br>{row['total']} papers")
    parents.append('Dataset')
    values.append(row['total'])
    colors.append('#64748B')
    
    # Included under this review
    ids.append(f"{row['review_id']}_incl")
    labels.append(f"Included<br><b>{int(row['included'])}</b>")
    parents.append(row['review_id'])
    values.append(int(row['included']))
    colors.append(COLORS['success'])
    
    # Excluded under this review
    ids.append(f"{row['review_id']}_excl")
    labels.append(f"Excluded<br><b>{int(row['excluded'])}</b>")
    parents.append(row['review_id'])
    values.append(int(row['excluded']))
    colors.append(COLORS['danger'])

fig3 = go.Figure(go.Treemap(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    marker=dict(
        colors=colors,
        line=dict(color='white', width=2)
    ),
    textfont=dict(size=11, color='white'),
    textposition='middle center',
    hovertemplate='<b>%{label}</b><br>Papers: %{value:,}<br>Share: %{percentRoot:.1%}<extra></extra>',
    branchvalues='total',
    pathbar=dict(visible=True, textfont=dict(size=12)),
    maxdepth=2,
))

fig3.update_layout(
    title=dict(
        text='<b>Ground Truth Dataset Structure</b><br>'
             '<span style="font-size:12px;color:#64748B">'
             'Click to drill down into reviews | Hover for details</span>',
        x=0.5,
        y=0.97,
        font=dict(size=16)
    ),
    height=600,
    font=dict(family='Segoe UI, sans-serif'),
    margin=dict(l=10, r=10, t=90, b=30),
    paper_bgcolor='white',
)

fig3.show()

# Save to HTML
html_path3 = f'{vis_dir}/ground_truth_treemap.html'
fig3.write_html(html_path3, include_plotlyjs='cdn')
print(f"✓ Saved visualization to: {html_path3}")

# ── SUMMARY STATISTICS BOX ──
print("\n" + "═"*60)
print("  GROUND TRUTH DATASET SUMMARY FOR PRESENTATION")
print("═"*60)
print(f"""
  📊 DATASET SIZE
     • Total paper-review pairs: {total_papers:,}
     • Unique systematic reviews: {n_reviews}
     
  🏷️ CLASS DISTRIBUTION  
     • Included (label=1): {n_included:,} ({n_included/total_papers*100:.1f}%)
     • Excluded (label=0): {n_excluded:,} ({n_excluded/total_papers*100:.1f}%)
     • Class ratio: 1:{n_excluded/n_included:.1f} (included:excluded)
     
  📈 PER-REVIEW STATISTICS
     • Avg papers per review: {total_papers/n_reviews:.0f}
     • Range: {review_stats['total'].min():.0f} - {review_stats['total'].max():.0f} papers
     • Avg inclusion rate: {avg_inclusion_rate*100:.1f}%
     
  📁 VISUALIZATIONS SAVED
     • {html_path}
     • {html_path2}  
     • {html_path3}
""")
print("═"*60)

✓ Saved visualization to: ../Visualisations/ground_truth_treemap.html

════════════════════════════════════════════════════════════
  GROUND TRUTH DATASET SUMMARY FOR PRESENTATION
════════════════════════════════════════════════════════════

  📊 DATASET SIZE
     • Total paper-review pairs: 10,225
     • Unique systematic reviews: 30
     
  🏷️ CLASS DISTRIBUTION  
     • Included (label=1): 2,265 (22.2%)
     • Excluded (label=0): 7,960 (77.8%)
     • Class ratio: 1:3.5 (included:excluded)
     
  📈 PER-REVIEW STATISTICS
     • Avg papers per review: 341
     • Range: 30 - 1299 papers
     • Avg inclusion rate: 21.6%
     
  📁 VISUALIZATIONS SAVED
     • ../Visualisations/ground_truth_overview.html
     • ../Visualisations/ground_truth_per_review.html  
     • ../Visualisations/ground_truth_treemap.html

════════════════════════════════════════════════════════════


## Prompt Selection Analysis

Interactive visualization showing how different prompt engineering strategies affected model performance across multiple metrics.

In [240]:
# ══════════════════════════════════════════════════════════════════════════════
#  INTERACTIVE PROMPT SELECTION ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
# Line graph showing how different prompts affect model performance
# - Select metric from dropdown
# - Click a line to highlight that model (others become transparent)

import json
import pandas as pd

# Load prompt testing results
prompt_results = pd.read_csv(f'{DATA_PATH}/results/step1_prompt_testing.csv')

# Calculate average metrics across 3 batches for each model-prompt combination
avg_results = prompt_results.groupby(['model', 'prompt']).agg({
    'acc': 'mean',
    'prec': 'mean', 
    'rec': 'mean',
    'f1': 'mean',
    'f2': 'mean'
}).reset_index()

# Create cleaner prompt labels
prompt_labels = {
    'p2_v1_baseline_obj_sel': 'Baseline',
    'p2_v2_no_role_obj_sel': 'No Role',
    'p2_v3_with_system_prompt_obj_sel': 'System Prompt',
    'p2_v4_minimal_obj_sel': 'Minimal',
    'p2_v5_detailed_obj_sel': 'Detailed'
}
avg_results['prompt_label'] = avg_results['prompt'].map(prompt_labels)

# Create cleaner model labels
model_labels = {
    'llama3_1_8b': 'Llama 3.1 8B',
    'mixtral_8x7b': 'Mixtral 8x7B',
    'gemma3_27b': 'Gemma 3 27B',
    'mistral_small3_2_24b': 'Mistral Small 3.2 24B'
}
avg_results['model_label'] = avg_results['model'].map(model_labels)

# Define prompt order
prompt_order = ['Baseline', 'No Role', 'System Prompt', 'Minimal', 'Detailed']
avg_results['prompt_order'] = avg_results['prompt_label'].map({p: i for i, p in enumerate(prompt_order)})
avg_results = avg_results.sort_values(['model', 'prompt_order'])

# Model colors
model_colors = {
    'Llama 3.1 8B': '#3B82F6',      # Blue
    'Mixtral 8x7B': '#10B981',       # Green
    'Gemma 3 27B': '#F59E0B',        # Amber
    'Mistral Small 3.2 24B': '#EF4444'  # Red
}

# Prepare data for JS
chart_data = {}
for model in avg_results['model_label'].unique():
    model_data = avg_results[avg_results['model_label'] == model].sort_values('prompt_order')
    chart_data[model] = {
        'prompts': model_data['prompt_label'].tolist(),
        'acc': (model_data['acc'] * 100).round(2).tolist(),
        'prec': (model_data['prec'] * 100).round(2).tolist(),
        'rec': (model_data['rec'] * 100).round(2).tolist(),
        'f1': (model_data['f1'] * 100).round(2).tolist(),
        'f2': (model_data['f2'] * 100).round(2).tolist(),
        'color': model_colors[model]
    }

# Prompt descriptions for the interactive panel with EXACT prompt text from 04_evaluate_llms.ipynb
prompt_descriptions = {
    'Baseline': {
        'title': 'V1: Baseline Prompt',
        'system_prompt': None,
        'prompt_text': '''You are assisting with title/abstract screening for a Cochrane systematic review.

=== REVIEW CONTEXT ===
Objectives: {objectives_text}

Selection criteria: {selection_criteria_text}

=== CANDIDATE PAPER ===
Title: {paper_title}

Abstract: {paper_abstract}

Using only the objectives and selection criteria above, should this paper be included in the review? In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable.

Respond with exactly one word: INCLUDE or EXCLUDE''',
        'key_features': ['Role framing ("You are assisting...")', 'Structured sections (=== headers ===)', 'Recall-biased instruction']
    },
    'No Role': {
        'title': 'V2: No Role Framing',
        'system_prompt': None,
        'prompt_text': '''=== REVIEW CONTEXT ===
Objectives: {objectives_text}

Selection criteria: {selection_criteria_text}

=== CANDIDATE PAPER ===
Title: {paper_title}

Abstract: {paper_abstract}

Using only the objectives and selection criteria above, should this paper be included in the review? In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable.

Respond with exactly one word: INCLUDE or EXCLUDE''',
        'key_features': ['No persona/role framing', 'Same structure as baseline', 'Direct task instruction']
    },
    'System Prompt': {
        'title': 'V3: With System Prompt',
        'system_prompt': '''You are an expert medical librarian assisting with title/abstract screening for Cochrane systematic reviews. Use only the review objectives and selection criteria in the user prompt. Always respond with exactly one word: INCLUDE or EXCLUDE.''',
        'prompt_text': '''=== REVIEW CONTEXT ===
Objectives: {objectives_text}

Selection criteria: {selection_criteria_text}

=== CANDIDATE PAPER ===
Title: {paper_title}

Abstract: {paper_abstract}

Should this paper be included in the review?

In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable. Respond with exactly one word: INCLUDE or EXCLUDE''',
        'key_features': ['Separate system message', 'Expert medical librarian role', 'Cleaner user message']
    },
    'Minimal': {
        'title': 'V4: Minimal Wording',
        'system_prompt': None,
        'prompt_text': '''Objectives: {objectives_text}

Selection criteria: {selection_criteria_text}

Paper title: {paper_title}
Paper abstract: {paper_abstract}

Use only the objectives and selection criteria. Respond with exactly one word: INCLUDE or EXCLUDE''',
        'key_features': ['Most concise variant', 'No framing or headers', 'Relies on model reasoning']
    },
    'Detailed': {
        'title': 'V5: Detailed/Step-by-Step',
        'system_prompt': None,
        'prompt_text': '''You are assisting with title/abstract screening for a Cochrane systematic review.

Step 1 — Understand the review scope from these fields only:
  Objectives: {objectives_text}
  Selection criteria: {selection_criteria_text}

Step 2 — Evaluate the candidate paper:
  Title: {paper_title}
  Abstract: {paper_abstract}

Step 3 — Decide whether the paper matches the review scope.
Step 4 — Give one-word output.
In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable. Respond with exactly one word: INCLUDE or EXCLUDE''',
        'key_features': ['Explicit step-by-step structure', 'Chain-of-thought style', 'Most verbose variant']
    }
}

prompt_descriptions_json = json.dumps(prompt_descriptions)
chart_data_json = json.dumps(chart_data)

# Build interactive HTML with prompt details panel
prompt_html = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Prompt Selection Analysis</title>
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{ 
            font-family: 'Segoe UI', system-ui, sans-serif; 
            background: #F8FAFC; 
            color: #1E293B;
            padding: 20px;
        }}
        .container {{ max-width: 1400px; margin: 0 auto; }}
        
        .controls {{
            display: flex;
            align-items: center;
            gap: 20px;
            margin-bottom: 20px;
            padding: 16px 20px;
            background: white;
            border-radius: 12px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}
        
        .control-group {{
            display: flex;
            align-items: center;
            gap: 10px;
        }}
        
        .control-label {{
            font-size: 13px;
            font-weight: 500;
            color: #64748B;
        }}
        
        select {{
            padding: 8px 12px;
            border: 2px solid #E2E8F0;
            border-radius: 8px;
            font-size: 14px;
            font-weight: 500;
            background: white;
            cursor: pointer;
            min-width: 150px;
        }}
        select:focus {{ outline: none; border-color: #3B82F6; }}
        
        .reset-btn {{
            padding: 8px 16px;
            background: #F1F5F9;
            border: none;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 500;
            color: #64748B;
            cursor: pointer;
            transition: all 0.2s;
        }}
        .reset-btn:hover {{ background: #E2E8F0; color: #475569; }}
        
        .main-content {{
            display: flex;
            gap: 24px;
        }}
        
        .chart-section {{
            flex: 1;
            background: white;
            border-radius: 12px;
            padding: 24px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}
        
        #chart {{ width: 100%; height: 400px; }}
        
        .prompt-labels {{
            display: flex;
            justify-content: space-between;
            padding: 0 50px;
            margin-top: 16px;
            gap: 8px;
        }}
        
        .prompt-label-btn {{
            flex: 1;
            padding: 10px 8px;
            background: #F8FAFC;
            border: 2px solid #E2E8F0;
            border-radius: 8px;
            font-size: 12px;
            font-weight: 600;
            color: #64748B;
            cursor: pointer;
            transition: all 0.2s;
            text-align: center;
            white-space: nowrap;
        }}
        .prompt-label-btn:hover {{
            border-color: #3B82F6;
            color: #3B82F6;
            background: #EFF6FF;
        }}
        .prompt-label-btn.active {{
            border-color: #3B82F6;
            color: white;
            background: #3B82F6;
        }}
        
        .legend {{
            display: flex;
            justify-content: center;
            gap: 24px;
            margin-top: 16px;
            flex-wrap: wrap;
        }}
        
        .legend-item {{
            display: flex;
            align-items: center;
            gap: 8px;
            padding: 8px 16px;
            border-radius: 8px;
            cursor: pointer;
            transition: all 0.2s;
            border: 2px solid transparent;
        }}
        .legend-item:hover {{ background: #F8FAFC; }}
        .legend-item.active {{ border-color: currentColor; background: #F8FAFC; }}
        .legend-item.faded {{ opacity: 0.3; }}
        
        .legend-color {{
            width: 24px;
            height: 4px;
            border-radius: 2px;
        }}
        
        .legend-label {{
            font-size: 13px;
            font-weight: 500;
        }}
        
        .instructions {{
            text-align: center;
            font-size: 12px;
            color: #94A3B8;
            margin-top: 12px;
        }}
        
        /* Prompt Details Panel */
        .prompt-panel {{
            width: 380px;
            flex-shrink: 0;
        }}
        
        .prompt-details {{
            background: white;
            border-radius: 12px;
            padding: 24px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            position: sticky;
            top: 20px;
            transition: all 0.3s ease;
        }}
        
        .prompt-placeholder {{
            text-align: center;
            padding: 60px 20px;
            color: #94A3B8;
        }}
        
        .placeholder-icon {{
            font-size: 48px;
            margin-bottom: 16px;
            opacity: 0.5;
        }}
        
        .placeholder-text {{
            font-size: 14px;
            line-height: 1.5;
        }}
        
        .prompt-title {{
            font-size: 18px;
            font-weight: 600;
            color: #1E293B;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 10px;
        }}
        
        .prompt-title-icon {{
            width: 28px;
            height: 28px;
            background: #3B82F6;
            border-radius: 6px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: white;
            font-size: 14px;
            font-weight: 700;
        }}
        
        .prompt-section {{
            margin-bottom: 20px;
        }}
        
        .prompt-section-title {{
            font-size: 12px;
            font-weight: 600;
            color: #64748B;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            margin-bottom: 10px;
        }}
        
        .prompt-text-box {{
            background: #1E293B;
            color: #E2E8F0;
            padding: 16px;
            border-radius: 8px;
            font-family: 'Consolas', 'Monaco', monospace;
            font-size: 12px;
            line-height: 1.6;
            white-space: pre-wrap;
            max-height: 300px;
            overflow-y: auto;
        }}
        
        .prompt-text-box .placeholder {{
            background: #3B82F6;
            color: white;
            padding: 2px 6px;
            border-radius: 4px;
            font-weight: 600;
        }}
        
        .prompt-features {{
            display: flex;
            flex-wrap: wrap;
            gap: 8px;
        }}
        
        .feature-tag {{
            background: #EFF6FF;
            color: #3B82F6;
            padding: 6px 12px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: 500;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="controls">
            <div class="control-group">
                <span class="control-label">Metric:</span>
                <select id="metric-select" onchange="updateChart()">
                    <option value="f1">F1 Score</option>
                    <option value="f2">F2 Score</option>
                    <option value="acc">Accuracy</option>
                    <option value="prec">Precision</option>
                    <option value="rec">Recall</option>
                </select>
            </div>
            <button class="reset-btn" onclick="resetHighlight()">Reset Highlight</button>
        </div>
        
        <div class="main-content">
            <div class="chart-section">
                <div id="chart"></div>
                <div id="prompt-labels" class="prompt-labels"></div>
                <div id="legend" class="legend"></div>
                <div class="instructions">Click on a model line or legend to highlight it</div>
            </div>
            
            <div class="prompt-panel">
                <div id="prompt-details" class="prompt-details">
                    <div class="prompt-placeholder">
                        <div class="placeholder-icon">📋</div>
                        <div class="placeholder-text">Click on a prompt label below the chart to see the prompt details</div>
                    </div>
                </div>
            </div>
        </div>
    </div>
    
    <script>
        const chartData = {chart_data_json};
        const promptDescriptions = {prompt_descriptions_json};
        const promptOrder = ['Baseline', 'No Role', 'System Prompt', 'Minimal', 'Detailed'];
        
        const metricLabels = {{
            'acc': 'Accuracy (%)',
            'prec': 'Precision (%)',
            'rec': 'Recall (%)',
            'f1': 'F1 Score (%)',
            'f2': 'F2 Score (%)'
        }};
        
        let highlightedModel = null;
        let selectedPrompt = null;
        
        function buildPromptLabels() {{
            const container = document.getElementById('prompt-labels');
            container.innerHTML = '';
            
            promptOrder.forEach(prompt => {{
                const btn = document.createElement('button');
                btn.className = 'prompt-label-btn';
                btn.textContent = prompt;
                btn.onclick = () => selectPrompt(prompt);
                container.appendChild(btn);
            }});
        }}
        
        function selectPrompt(prompt) {{
            if (selectedPrompt === prompt) {{
                selectedPrompt = null;
            }} else {{
                selectedPrompt = prompt;
            }}
            updatePromptLabels();
            updatePromptDetails();
        }}
        
        function updatePromptLabels() {{
            const btns = document.querySelectorAll('.prompt-label-btn');
            btns.forEach((btn, i) => {{
                const prompt = promptOrder[i];
                btn.classList.toggle('active', selectedPrompt === prompt);
            }});
        }}
        
        function updatePromptDetails() {{
            const panel = document.getElementById('prompt-details');
            
            if (!selectedPrompt) {{
                panel.innerHTML = `
                    <div class="prompt-placeholder">
                        <div class="placeholder-icon">📋</div>
                        <div class="placeholder-text">Click on a prompt label below the chart to see the prompt details</div>
                    </div>
                `;
                return;
            }}
            
            const info = promptDescriptions[selectedPrompt];
            const featuresHtml = info.key_features.map(f => `<span class="feature-tag">${{f}}</span>`).join('');
            
            // Format prompt text with highlighted placeholders
            const formattedPrompt = info.prompt_text
                .replace(/\\{{objectives_text\\}}/g, '<span class="placeholder">{{objectives_text}}</span>')
                .replace(/\\{{selection_criteria_text\\}}/g, '<span class="placeholder">{{selection_criteria_text}}</span>')
                .replace(/\\{{paper_title\\}}/g, '<span class="placeholder">{{paper_title}}</span>')
                .replace(/\\{{paper_abstract\\}}/g, '<span class="placeholder">{{paper_abstract}}</span>');
            
            // Format system prompt if present
            let systemHtml = '';
            if (info.system_prompt) {{
                const formattedSystem = info.system_prompt;
                systemHtml = `
                    <div class="prompt-section">
                        <div class="prompt-section-title">System Message</div>
                        <div class="prompt-text-box" style="background: #1E3A5F; border-left: 3px solid #3B82F6;">${{formattedSystem}}</div>
                    </div>
                `;
            }}
            
            panel.innerHTML = `
                <div class="prompt-title">
                    <span class="prompt-title-icon">P</span>
                    ${{info.title}}
                </div>
                
                ${{systemHtml}}
                
                <div class="prompt-section">
                    <div class="prompt-section-title">${{info.system_prompt ? 'User Message' : 'Prompt Template'}}</div>
                    <div class="prompt-text-box">${{formattedPrompt}}</div>
                </div>
                
                <div class="prompt-section">
                    <div class="prompt-section-title">Key Features</div>
                    <div class="prompt-features">${{featuresHtml}}</div>
                </div>
            `;
        }}
        
        function buildLegend() {{
            const legend = document.getElementById('legend');
            legend.innerHTML = '';
            
            Object.keys(chartData).forEach(model => {{
                const item = document.createElement('div');
                item.className = 'legend-item';
                item.style.color = chartData[model].color;
                item.innerHTML = `
                    <div class="legend-color" style="background: ${{chartData[model].color}}"></div>
                    <span class="legend-label">${{model}}</span>
                `;
                item.onclick = () => highlightModel(model);
                legend.appendChild(item);
            }});
        }}
        
        function updateChart() {{
            const metric = document.getElementById('metric-select').value;
            const traces = [];
            
            Object.keys(chartData).forEach(model => {{
                const data = chartData[model];
                const opacity = highlightedModel === null ? 1 : (highlightedModel === model ? 1 : 0.15);
                const width = highlightedModel === model ? 4 : 2;
                
                traces.push({{
                    x: data.prompts,
                    y: data[metric],
                    name: model,
                    type: 'scatter',
                    mode: 'lines+markers',
                    line: {{ color: data.color, width: width }},
                    marker: {{ size: 10, color: data.color }},
                    opacity: opacity,
                    hovertemplate: `<b>${{model}}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>`
                }});
            }});
            
            const layout = {{
                title: {{
                    text: 'Model Performance Across Prompt Variants',
                    font: {{ size: 18, color: '#1E293B' }}
                }},
                xaxis: {{
                    title: '',
                    tickfont: {{ size: 1, color: 'transparent' }},
                    gridcolor: '#E2E8F0'
                }},
                yaxis: {{
                    title: metricLabels[metric],
                    gridcolor: '#E2E8F0',
                    range: [0, 100]
                }},
                plot_bgcolor: 'white',
                paper_bgcolor: 'white',
                showlegend: false,
                hovermode: 'closest',
                margin: {{ t: 60, b: 40, l: 60, r: 30 }}
            }};
            
            Plotly.react('chart', traces, layout, {{responsive: true}});
            
            // Add click handler
            document.getElementById('chart').on('plotly_click', function(data) {{
                const model = data.points[0].data.name;
                highlightModel(model);
            }});
        }}
        
        function highlightModel(model) {{
            if (highlightedModel === model) {{
                highlightedModel = null;
            }} else {{
                highlightedModel = model;
            }}
            updateChart();
            updateLegendHighlight();
        }}
        
        function updateLegendHighlight() {{
            const items = document.querySelectorAll('.legend-item');
            items.forEach(item => {{
                const label = item.querySelector('.legend-label').textContent;
                item.classList.remove('active', 'faded');
                if (highlightedModel) {{
                    if (label === highlightedModel) {{
                        item.classList.add('active');
                    }} else {{
                        item.classList.add('faded');
                    }}
                }}
            }});
        }}
        
        function resetHighlight() {{
            highlightedModel = null;
            updateChart();
            updateLegendHighlight();
        }}
        
        // Initialize
        buildPromptLabels();
        buildLegend();
        updateChart();
    </script>
</body>
</html>
'''

# Save interactive HTML
html_path_prompt = f'{vis_dir}/prompt_selection_analysis.html'
with open(html_path_prompt, 'w', encoding='utf-8') as f:
    f.write(prompt_html)

print(f"✓ Saved interactive visualization to: {html_path_prompt}")
print(f"  → Select metric from dropdown (F1, F2, Accuracy, Precision, Recall)")
print(f"  → Click a line or legend item to highlight that model")
print(f"  → Click a prompt label to see details")

# Also show inline Plotly chart for notebook preview
import plotly.graph_objects as go

# Create traces for each metric (we'll toggle visibility with dropdown)
metrics = ['f1', 'f2', 'acc', 'prec', 'rec']
metric_titles = {
    'f1': 'F1 Score (%)',
    'f2': 'F2 Score (%)', 
    'acc': 'Accuracy (%)',
    'prec': 'Precision (%)',
    'rec': 'Recall (%)'
}

fig_prompt = go.Figure()

# Add traces for default metric (f1)
for model in avg_results['model_label'].unique():
    model_data = avg_results[avg_results['model_label'] == model].sort_values('prompt_order')
    fig_prompt.add_trace(go.Scatter(
        x=model_data['prompt_label'],
        y=model_data['f1'] * 100,
        name=model,
        mode='lines+markers',
        line=dict(color=model_colors[model], width=2),
        marker=dict(size=10),
        hovertemplate=f'<b>{model}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>'
    ))

# Create dropdown buttons
dropdown_buttons = []
for metric in metrics:
    y_values = []
    for model in avg_results['model_label'].unique():
        model_data = avg_results[avg_results['model_label'] == model].sort_values('prompt_order')
        y_values.append(model_data[metric].values * 100)
    
    dropdown_buttons.append(dict(
        label=metric_titles[metric],
        method='update',
        args=[
            {'y': y_values},
            {'yaxis.title': metric_titles[metric]}
        ]
    ))

fig_prompt.update_layout(
    title=dict(
        text='Model Performance Across Prompt Variants',
        font=dict(size=18, color='#1E293B')
    ),
    xaxis_title='',
    yaxis_title='F1 Score (%)',
    xaxis=dict(tickfont=dict(size=12)),
    yaxis=dict(range=[0, 100], gridcolor='#E2E8F0'),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5,
        itemclick='toggle',
        itemdoubleclick='toggleothers'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='x unified',
    margin=dict(b=100),
    updatemenus=[
        dict(
            active=0,
            buttons=dropdown_buttons,
            direction='down',
            showactive=True,
            x=1.0,
            xanchor='right',
            y=1.15,
            yanchor='top',
            bgcolor='white',
            bordercolor='#E2E8F0',
            font=dict(size=12)
        )
    ],
    annotations=[
        dict(
            text='Metric:',
            x=0.85,
            y=1.17,
            xref='paper',
            yref='paper',
            showarrow=False,
            font=dict(size=12, color='#64748B')
        )
    ]
)

fig_prompt.show()

✓ Saved interactive visualization to: ../Visualisations/prompt_selection_analysis.html
  → Select metric from dropdown (F1, F2, Accuracy, Precision, Recall)
  → Click a line or legend item to highlight that model
  → Click a prompt label to see details


## 📋 Summary

This dashboard visualizes the outputs of **Notebooks 00 and 01** in the data extraction pipeline:

| Metric | Value |
|--------|-------|
| **Total Cochrane Reviews** | Extracted from PDFs |
| **Reference Categories** | Included, Excluded, Additional, Awaiting, Ongoing, Other Versions |
| **Abstract Sources** | PubMed (primary), CrossRef (fallback) |

### Key Insights
- **Included** references are the studies that met the Cochrane review's inclusion criteria
- **Excluded** references are studies that were screened but did not meet criteria
- **Additional** references typically include background/methodology citations
- Abstract fetching prioritizes **Included** references for the screening evaluation task

---
*Dashboard generated from extracted pipeline data*